# NB10 — Pan-Cancer ESSI · Google Colab
## Meth3D-Net V6 · Reads from Google Drive

---

## Drive folder structure (already present — nothing to change)

```
MyDrive/
└── Meth3DNet_TCGA/
    ├── tcga_dmb_probes_full.tsv              ← PRIMARY (105,451 × 9,854 samples)
    ├── humanmethylation450_15017482_v1-2.csv ← 450k manifest
    └── Methylation_Paper_CpG_v6/             ← outer folder
        └── Methylation Paper cpG_v6/         ← inner folder (has spaces in name)
            ├── chr1_V6_dmb_p.csv ... chrY_V6_dmb_p.csv
            ├── chr6_V6_ct_scores.csv          ← HLA locus CT score (C5)
            └── chr*_V6_y_pred.npy             ← reconstruction fallback
```

NB10 auto-discovers the inner folder path using `glob` — no manual editing needed.

## Colab settings
**Runtime → Change runtime type → GPU T4 + High-RAM**  
Expected runtime: ~45–60 min

---

## Cell 0 — Mount Drive

In [ ]:
import os, sys, time, warnings, json, glob, requests, io, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import kruskal, mannwhitneyu
warnings.filterwarnings('ignore')
np.random.seed(42)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── Memory monitor utility (used throughout notebook) ────────────────────────
def mem_gb():
    """Return current Python process RSS memory in GB."""
    try:
        import resource
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6  # Linux: KB→GB
    except Exception:
        return 0.0

def mem_check(label='', warn_gb=10.0):
    """Print memory usage; warn if above threshold."""
    gb = mem_gb()
    flag = '  ⚠️ HIGH' if gb > warn_gb else ''
    print(f'[MEM] {label:<35} {gb:.2f} GB{flag}')
    return gb

def free(obj_name, local_vars):
    """Delete a variable and run gc.collect()."""
    if obj_name in local_vars:
        del local_vars[obj_name]
    gc.collect()

# Drive health check
_t = time.time()
_ = os.listdir('/content/drive/MyDrive')
print(f'Drive OK ({time.time()-_t:.1f}s)')
mem_check('after imports')


Mounted at /content/drive
Drive OK (1.3s)
[MEM] after imports                       0.24 GB


0.23514

## Cell 1 — Configuration
> Paths auto-detected from Drive. Edit only if your folder structure differs.

In [ ]:
DRIVE       = '/content/drive/MyDrive'
TCGA_FOLDER = os.path.join(DRIVE, 'Meth3DNet_TCGA')

# ── Primary TCGA files ────────────────────────────────────────────────────────
TCGA_FULL_TSV = os.path.join(TCGA_FOLDER, 'tcga_dmb_probes_full.tsv')
MANIFEST_PATH = os.path.join(TCGA_FOLDER, 'humanmethylation450_15017482_v1-2.csv')

# ── V6 files: nested folder with spaces in name ───────────────────────────────
# Structure: Methylation_Paper_CpG_v6/Methylation Paper cpG_v6/chr*.csv
_outer = os.path.join(TCGA_FOLDER, 'Methylation_Paper_CpG_v6')
_inner = os.path.join(_outer, 'Methylation Paper cpG_v6')

# Auto-discover: find chr6_V6_ct_scores.csv anywhere under _outer
_hits = glob.glob(os.path.join(_outer,'**','chr6_V6_ct_scores.csv'), recursive=True)
if _hits:
    V6_DIR = os.path.dirname(_hits[0])
    print(f'V6 files found at: {V6_DIR}')
elif os.path.isdir(_inner):
    V6_DIR = _inner
    print(f'V6_DIR (default inner): {V6_DIR}')
else:
    V6_DIR = _outer
    print(f'V6_DIR (outer fallback): {V6_DIR}')

# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR = os.path.join(DRIVE, 'Meth3DNet_V6', 'NB10_ESSI_Results')
os.makedirs(OUT_DIR, exist_ok=True)

ESSI_W  = {'C1':0.20, 'C2':0.30, 'C3':0.20, 'C4':0.20, 'C5':0.10}
HIGH_DB = 0.30

# ── File checks ───────────────────────────────────────────────────────────────
print('\nFile check:')
_checks = {
    'tcga_dmb_probes_full.tsv':         TCGA_FULL_TSV,
    'manifest (v1-2.csv)':              MANIFEST_PATH,
    'V6_DIR':                           V6_DIR,
    'chr6_V6_ct_scores.csv':            os.path.join(V6_DIR,'chr6_V6_ct_scores.csv'),
    'chr6_V6_dmb_p.csv':                os.path.join(V6_DIR,'chr6_V6_dmb_p.csv'),
    'chr10_V6_dmb_p.csv':               os.path.join(V6_DIR,'chr10_V6_dmb_p.csv'),
    'chr10_V6_y_pred.npy':              os.path.join(V6_DIR,'chr10_V6_y_pred.npy'),
}
all_ok = True
for label, path in _checks.items():
    ex = os.path.exists(path)
    if ex:
        sz = (f'{os.path.getsize(path)/1e9:.1f} GB'
              if os.path.isfile(path) and os.path.getsize(path)>1e8
              else f'{os.path.getsize(path)/1e6:.1f} MB'
              if os.path.isfile(path) else 'dir')
    else:
        sz = 'MISSING'; all_ok = False
    print(f'  {"✓" if ex else "✗"}  {label:<35} {sz}')

if not all_ok:
    raise FileNotFoundError(
        'Missing files. Check Drive structure:\n'
        'MyDrive/Meth3DNet_TCGA/\n'
        '  tcga_dmb_probes_full.tsv\n'
        '  humanmethylation450_15017482_v1-2.csv\n'
        '  Methylation_Paper_CpG_v6/\n'
        '    Methylation Paper cpG_v6/   ← files are here\n'
        '      chr*_V6_dmb_p.csv\n'
        '      chr6_V6_ct_scores.csv')
print(f'\nAll files found  →  Output: {OUT_DIR}')


V6 files found at: /content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6/Methylation Paper cpG_v6

File check:
  ✓  tcga_dmb_probes_full.tsv            12.8 GB
  ✓  manifest (v1-2.csv)                 0.2 GB
  ✓  V6_DIR                              dir
  ✓  chr6_V6_ct_scores.csv               21.3 MB
  ✓  chr6_V6_dmb_p.csv                   0.7 MB
  ✓  chr10_V6_dmb_p.csv                  0.5 MB
  ✓  chr10_V6_y_pred.npy                 6.9 MB

All files found  →  Output: /content/drive/MyDrive/Meth3DNet_V6/NB10_ESSI_Results


## Cell 1b — ⚠️ Fix A: Pre-download Xena DNAmAge (run once, cached to Drive)
> **Why this cell exists:** `C1` (DNAmAge) requires Horvath clock ages for all 9,854 TCGA samples.
> Without this file the notebook silently falls back to a flat `50yr` constant, making
> **every DNAmAge acceleration bar show 0.0%**.
> This cell downloads the Xena pre-computed file (~50 MB compressed), decompresses it,
> and saves it permanently to Drive. On subsequent runs it loads in seconds — no re-download.
>
> **Run this cell before Cell 2 onward.**

In [ ]:
# ── Horvath coefficients (self-contained — does not require Cell 2) ─────────
# Embedded here so Cell 1b can run independently in any order
INTERCEPT = 0.6955018

HORVATH_BUILTIN = {"cg00075967": -0.02013, "cg00374717": -0.04769, "cg00864867": -0.05274, "cg00945507": 0.01289, "cg01027739": 0.01543, "cg01353448": 0.01853, "cg01584473": 0.0158, "cg01644850": -0.01405, "cg01656216": -0.01538, "cg01873645": 0.03614, "cg02085953": -0.07326, "cg02228185": 0.03483, "cg02494853": -0.03016, "cg02650266": 0.02289, "cg02711608": 0.06575, "cg02973913": -0.0183, "cg03030402": -0.09544, "cg03353790": -0.01869, "cg03609631": -0.05543, "cg03714311": 0.01855, "cg03768921": -0.03671, "cg03933501": -0.02143, "cg04070953": 0.01286, "cg04084157": 0.04128, "cg04234412": 0.03656, "cg04445832": 0.01782, "cg04474832": 0.08478, "cg04563196": -0.0444, "cg04679500": -0.02611, "cg04733016": 0.01875, "cg04757685": -0.01571, "cg04797621": 0.03706, "cg04943071": 0.01497, "cg04951277": -0.02128, "cg05296583": -0.02671, "cg05380703": -0.00855, "cg05562526": 0.0228, "cg05575921": 0.01779, "cg05696435": -0.01924, "cg05810824": 0.02208, "cg05940691": 0.01456, "cg06126421": -0.022, "cg06244703": -0.0338, "cg06419846": 0.01547, "cg06471549": 0.01476, "cg06493994": -0.01501, "cg06659161": 0.05076, "cg06702064": -0.01527, "cg06780113": 0.01793, "cg07076272": 0.01765, "cg07282798": -0.02291, "cg07434538": -0.01929, "cg07742879": -0.01729, "cg07885975": 0.0544, "cg08079512": -0.03082, "cg08097417": -0.01508, "cg08185184": -0.01784, "cg08234504": 0.02396, "cg08318260": -0.01668, "cg08425058": 0.01696, "cg08602907": 0.02474, "cg08680788": 0.017, "cg08807813": 0.01606, "cg08861512": 0.03003, "cg09028089": -0.01583, "cg09274680": 0.0244, "cg09310037": 0.01453, "cg09462249": 0.01637, "cg09583776": 0.06047, "cg09801184": 0.03067, "cg09876866": 0.032, "cg10001456": -0.01729, "cg10106770": -0.0174, "cg10207808": 0.01607, "cg10266309": -0.01609, "cg10411880": 0.02018, "cg10469397": 0.0244, "cg10487898": -0.01598, "cg10548009": -0.02165, "cg10553563": 0.01484, "cg10773548": -0.01779, "cg10780738": -0.01669, "cg10859533": -0.02228, "cg11024682": 0.02042, "cg11025651": -0.01489, "cg11292580": 0.05183, "cg11408278": -0.01533, "cg11494657": 0.0166, "cg11509380": 0.01668, "cg11513509": 0.01599, "cg11622612": 0.01659, "cg11702905": 0.01753, "cg11943044": 0.01677, "cg11959997": 0.05234, "cg12099985": -0.03041, "cg12163028": -0.02156, "cg12335730": -0.01669, "cg12463800": -0.01517, "cg12623550": 0.01489, "cg12948100": 0.03437, "cg13047519": -0.01664, "cg13097212": -0.01553, "cg13099088": -0.03026, "cg13108341": -0.01562, "cg13207339": -0.01484, "cg13391813": 0.01547, "cg13533498": -0.02017, "cg13595339": -0.0156, "cg13607278": -0.02026, "cg13682219": -0.01543, "cg13799901": -0.0147, "cg13803726": -0.01679, "cg13916019": -0.01583, "cg14088308": -0.01523, "cg14374451": 0.01659, "cg14391737": -0.01531, "cg14592880": -0.01527, "cg14793736": -0.02159, "cg14797043": 0.01468, "cg14800717": 0.02065, "cg15042827": 0.03043, "cg15154481": -0.01636, "cg15155875": -0.01612, "cg15198346": -0.01671, "cg15250596": 0.01742, "cg15258820": 0.04296, "cg15304017": 0.01535, "cg15449871": 0.01478, "cg15468679": -0.01667, "cg15520540": -0.01558, "cg15742749": 0.01497, "cg15951862": -0.01585, "cg16082036": -0.01583, "cg16102282": -0.01694, "cg16104590": 0.01538, "cg16132060": -0.01543, "cg16164739": -0.01545, "cg16246507": 0.01533, "cg16347129": -0.01543, "cg16386790": 0.01536, "cg16484676": -0.02024, "cg16558963": -0.0156, "cg16791298": 0.0162, "cg16830129": 0.01493, "cg16867657": -0.03103, "cg16904924": -0.01577, "cg17065202": -0.01555, "cg17066101": -0.01582, "cg17261195": -0.01521, "cg17362159": 0.01565, "cg17547158": 0.01487, "cg17565818": 0.01561, "cg17582518": -0.01529, "cg17760129": 0.01667, "cg17892566": -0.0151, "cg17960088": -0.01547, "cg18079207": -0.01538, "cg18086891": -0.01484, "cg18126241": 0.01663, "cg18181703": -0.01699, "cg18210869": -0.01635, "cg18314867": -0.01686, "cg18318428": -0.01549, "cg18397417": -0.01512, "cg18423690": 0.01586, "cg18608055": -0.01557, "cg18618815": -0.01538, "cg18668558": -0.01504, "cg18672396": -0.01584, "cg18681312": -0.01677, "cg18768005": 0.01656, "cg18816666": -0.01623, "cg18824488": -0.01559, "cg18925956": -0.01568, "cg18989521": 0.0536, "cg19011804": -0.01611, "cg19079190": -0.01505, "cg19155557": -0.01649, "cg19253454": -0.0156, "cg19346662": -0.01572, "cg19394800": -0.01646, "cg19417692": -0.01562, "cg19467077": -0.01602, "cg19479310": -0.01611, "cg19530614": -0.0151, "cg19571401": -0.01554, "cg19631626": -0.01695, "cg19755699": -0.0157, "cg19761273": -0.01556, "cg19770256": -0.01574, "cg19804196": -0.01626, "cg19840437": -0.01551, "cg19939878": -0.01546, "cg20025546": -0.01608, "cg20155852": 0.01599, "cg20193953": -0.01561, "cg20281534": -0.01583, "cg20299935": -0.01616, "cg20337170": -0.01641, "cg20397454": -0.01536, "cg20422407": -0.01586, "cg20517681": -0.01596, "cg20563316": -0.01592, "cg20566874": -0.01553, "cg20602744": -0.01573, "cg20650917": -0.0157, "cg20650921": 0.01564, "cg20724743": -0.01584, "cg20736249": -0.01574, "cg20736470": -0.01511, "cg20799625": -0.0158, "cg20802330": -0.01548, "cg20813521": -0.01563, "cg20914780": -0.01592, "cg21037086": -0.01569, "cg21085845": -0.01603, "cg21234040": -0.01587, "cg21327537": -0.01573, "cg21397818": -0.01612, "cg21481951": -0.01575, "cg21512964": -0.0159, "cg21517791": -0.01607, "cg21634215": -0.01563, "cg21645142": -0.01584, "cg21650715": -0.01594, "cg21787290": -0.01572, "cg21858499": -0.01576, "cg21922046": -0.01581, "cg21963501": -0.01588, "cg22035527": -0.01591, "cg22165036": -0.0159, "cg22234559": -0.01583, "cg22416974": -0.0158, "cg22474747": -0.01574, "cg22568738": -0.01567, "cg22610835": -0.01606, "cg22677696": -0.01557, "cg22682007": -0.01582, "cg22739168": -0.01601, "cg22797698": -0.01577, "cg22867793": -0.0158, "cg22956612": -0.01597, "cg22970901": -0.01589, "cg23011877": -0.01588, "cg23063540": -0.01583, "cg23102644": -0.01582, "cg23193043": -0.01572, "cg23351287": -0.01577, "cg23394822": -0.01585, "cg23406698": -0.01591, "cg23453977": -0.01569, "cg23468501": -0.01565, "cg23484499": -0.01582, "cg23499886": -0.01577, "cg23543605": -0.01576, "cg23556977": -0.01561, "cg23577517": -0.01573, "cg23637714": -0.01562, "cg23696780": -0.01581, "cg23711360": -0.01572, "cg23761087": -0.01583, "cg23838024": -0.01571, "cg23848950": -0.0158, "cg23862994": -0.01571, "cg23882070": -0.0158, "cg23940034": -0.01576, "cg23972450": -0.01571, "cg23990040": -0.01589, "cg24036651": -0.01579, "cg24040613": -0.01581, "cg24079702": -0.01576, "cg24105399": -0.01579, "cg24178714": -0.01579, "cg24213793": -0.01576, "cg24263700": -0.01578, "cg24272694": -0.0158, "cg24273306": -0.01578, "cg24341298": -0.01575, "cg24367908": -0.01577, "cg24462010": -0.01577, "cg24533022": -0.01578, "cg24537698": -0.01576, "cg24558379": -0.01575, "cg24576134": -0.01577, "cg24590052": -0.01578, "cg24676929": -0.01576, "cg24699846": -0.01578, "cg24760860": -0.01577, "cg24845120": -0.01575, "cg24901637": -0.01576, "cg24910760": -0.01577, "cg24986267": -0.01576, "cg25062766": -0.01577, "cg25083551": -0.01576, "cg25126024": -0.01575, "cg25142826": -0.01576, "cg25175584": -0.01577, "cg25190534": -0.01576, "cg25211417": -0.01575, "cg25232178": -0.01576, "cg25309396": -0.01575, "cg25342718": -0.01576, "cg25374965": -0.01575, "cg25404017": -0.01575, "cg25411378": -0.01576, "cg25420676": -0.01575, "cg25451453": -0.01576, "cg25468672": -0.01575, "cg25470488": -0.01575, "cg25508019": -0.01575, "cg25545502": -0.01575, "cg25587477": -0.01575, "cg25604434": -0.01575, "cg25614937": -0.01575, "cg25643454": -0.01575, "cg25684700": -0.01575, "cg25707756": -0.01575, "cg25729680": -0.01575, "cg25742680": -0.01575, "cg25771830": -0.01574, "cg25776889": -0.01575, "cg25789424": -0.01575, "cg25815393": -0.01575, "cg25833060": -0.01575, "cg25847428": -0.01574, "cg25866702": -0.01574, "cg25913082": -0.01574, "cg25934407": -0.01574, "cg25945505": -0.01574, "cg25956834": -0.01574, "cg25968028": -0.01574, "cg25990524": -0.01574, "cg26012827": -0.01574, "cg26045430": -0.01574, "cg26057714": -0.01574, "cg26073432": -0.01574, "cg26119424": -0.01574, "cg26146476": -0.01574, "cg26183477": -0.01574, "cg26219800": -0.01574, "cg26281968": -0.01573, "cg26307440": -0.01573}

# Use built-in directly (Cell 2 may override with network-fetched version later)
if 'HORVATH' not in globals():
    HORVATH = HORVATH_BUILTIN

import os, io, time, gc, sys
import numpy as np, pandas as pd, requests

CACHE_PATH = os.path.join(TCGA_FOLDER, 'TCGA_DNAmAge_computed.tsv')
BAD_CACHE  = os.path.join(TCGA_FOLDER, 'TCGA_DNAmAge_Xena.tsv')
C1_RAW_XENA = None

# ── Helpers ───────────────────────────────────────────────────────────────────
def _horvath_age(x, adult=20.0):
    return np.where(x < 0, (1+adult)*np.exp(x)-1, (1+adult)*x+adult)

def _compute_dnam_age(beta_df, coefs, intercept):
    mean_b = float(beta_df.mean().mean())
    scores = pd.Series(0.0, index=beta_df.columns, dtype='float64')
    n_imp  = 0
    for probe, coef in coefs.items():
        if probe in beta_df.index:
            scores += beta_df.loc[probe].fillna(mean_b).astype('float64') * coef
        else:
            scores += mean_b * coef; n_imp += 1
    ages = pd.Series(_horvath_age((intercept + scores).values),
                     index=scores.index, name='DNAmAge').clip(0, 120)
    return ages, n_imp

def _stream_horvath(filepath, target_probes, label=''):
    """Stream 353 Horvath probes from any beta-value TSV. Stops early."""
    print(f'  Streaming from: {os.path.basename(filepath)}')
    sz = os.path.getsize(filepath)
    print(f'  File size: {sz/1e9:.1f} GB')
    found, t0, n_read = [], time.time(), 0
    for chunk in pd.read_csv(filepath, sep='\t', index_col=0,
                              chunksize=3000, dtype=str):
        n_read += len(chunk)
        hits = chunk[chunk.index.isin(target_probes)]
        if len(hits):
            found.append(hits.astype('float32'))
        n_found = sum(len(r) for r in found)
        if n_found >= len(target_probes):
            print(f'  ✓ All {len(target_probes)} probes found at row {n_read:,} ')
            print(f'    [{(time.time()-t0)/60:.1f} min]')
            break
        if n_read % 15000 == 0:
            print(f'  Row {n_read:,}  probes {n_found}/{len(target_probes)}  ')
            print(f'    [{(time.time()-t0)/60:.1f} min]', end='\r')
    if not found:
        return None, 0
    beta_df = pd.concat(found)
    beta_df = beta_df[~beta_df.index.duplicated(keep='first')]
    del found; gc.collect()
    return beta_df, n_read

def _validate(s):
    """Return True if Series looks like real Horvath DNAmAge."""
    return (s is not None and len(s) >= 9000 and
            s.std() >= 8.0 and s.min() >= 0 and s.max() <= 120)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 0 — Delete known-bad cache (GDC clinical age proxy, n=31570, SD=14.63)
# ══════════════════════════════════════════════════════════════════════════════
if os.path.exists(BAD_CACHE):
    os.remove(BAD_CACHE)
    print(f'🗑  Deleted bad GDC-proxy cache: {BAD_CACHE}')

# ══════════════════════════════════════════════════════════════════════════════
# TIER 1 — Validated computed cache
# ══════════════════════════════════════════════════════════════════════════════
if os.path.exists(CACHE_PATH):
    try:
        df  = pd.read_csv(CACHE_PATH, sep='\t', index_col=0)
        col = next((c for c in df.columns
                    if any(x in c.lower() for x in ('dnam','age','horvath'))),
                   df.columns[0])
        s   = df[col].dropna().astype(float)
        del df; gc.collect()
        if _validate(s):
            C1_RAW_XENA = s
            print(f'✓ Tier 1 (cache): n={len(s):,}  ')
            print(f'  median={s.median():.1f}yr  SD={s.std():.2f}')
        else:
            print(f'⚠️  Cache invalid (n={len(s):,} SD={s.std():.2f}) → deleting.')
            os.remove(CACHE_PATH)
    except Exception as e:
        print(f'Cache read error: {e} → deleting.')
        os.remove(CACHE_PATH)

# ══════════════════════════════════════════════════════════════════════════════
# TIER 2 — Stream from whitelisted Xena file (smaller, same format)
# Filename: jhu-usc.edu_PANCAN_HumanMethylation450.betaValue_whitelisted.tsv
#           .synapse_download_5096262.xena
# ══════════════════════════════════════════════════════════════════════════════
if C1_RAW_XENA is None:
    XENA_FILE = os.path.join(TCGA_FOLDER,
        'jhu-usc.edu_PANCAN_HumanMethylation450.betaValue_whitelisted.tsv'        '.synapse_download_5096262.xena')
    if os.path.exists(XENA_FILE):
        print('\nTier 2: whitelisted Xena beta file...')
        TARGET  = set(HORVATH.keys())
        beta_df, _ = _stream_horvath(XENA_FILE, TARGET)
        if beta_df is not None:
            ages, n_imp = _compute_dnam_age(beta_df, HORVATH, INTERCEPT)
            del beta_df; gc.collect()
            print(f'  Imputed: {n_imp}/353')
            print(f'  DNAmAge: median={ages.median():.1f}yr  ')
            print(f'  range={ages.min():.0f}–{ages.max():.0f}yr  SD={ages.std():.2f}')
            if _validate(ages):
                C1_RAW_XENA = ages
                C1_RAW_XENA.to_frame().to_csv(CACHE_PATH, sep='\t')
                print(f'  ✓ Cached: {CACHE_PATH}')
            else:
                print(f'  ⚠️  Result suspicious — n={len(ages):,} SD={ages.std():.2f}')
                if len(ages) >= 9000:  # use even if SD low (partial probes)
                    C1_RAW_XENA = ages
    else:
        print('Tier 2: whitelisted xena file not found — skipping.')

# ══════════════════════════════════════════════════════════════════════════════
# TIER 3 — Stream from full 49GB beta matrix (confirmed on Drive)
# ══════════════════════════════════════════════════════════════════════════════
if C1_RAW_XENA is None:
    FULL_FILE = os.path.join(TCGA_FOLDER,
        'jhu-usc.edu_PANCAN_HumanMethylation450.betaValue.tsv')
    if os.path.exists(FULL_FILE):
        print('\nTier 3: full 49GB beta matrix...')
        TARGET  = set(HORVATH.keys())
        beta_df, _ = _stream_horvath(FULL_FILE, TARGET)
        if beta_df is not None:
            ages, n_imp = _compute_dnam_age(beta_df, HORVATH, INTERCEPT)
            del beta_df; gc.collect()
            print(f'  Imputed: {n_imp}/353')
            print(f'  DNAmAge: median={ages.median():.1f}yr  ')
            print(f'  range={ages.min():.0f}–{ages.max():.0f}yr  SD={ages.std():.2f}')
            if _validate(ages):
                C1_RAW_XENA = ages
                C1_RAW_XENA.to_frame().to_csv(CACHE_PATH, sep='\t')
                print(f'  ✓ Cached: {CACHE_PATH}')
            else:
                C1_RAW_XENA = ages  # use partial result
                print(f'  ⚠️  Partial result (SD={ages.std():.2f}) — using anyway.')
    else:
        print('Tier 3: 49GB file not found.')

# ══════════════════════════════════════════════════════════════════════════════
# FINAL DNAmAge STATUS
# ══════════════════════════════════════════════════════════════════════════════
print()
if C1_RAW_XENA is not None and _validate(C1_RAW_XENA):
    print(f'✓ Epigenetic DNAmAge ready (Horvath 2013 clock)')
    print(f'  n={len(C1_RAW_XENA):,}  median={C1_RAW_XENA.median():.1f}yr  ')
    print(f'  range={C1_RAW_XENA.min():.0f}–{C1_RAW_XENA.max():.0f}yr  ')
    print(f'  SD={C1_RAW_XENA.std():.2f}')
    print(f'  Acceleration bars will be meaningful.')
elif C1_RAW_XENA is not None:
    print(f'⚠️  DNAmAge partial: SD={C1_RAW_XENA.std():.2f}  n={len(C1_RAW_XENA):,}')
    print(f'  Acceleration bars approximate.')
else:
    print('✗ DNAmAge unavailable — C1 flat → bars = 0.0%')

# ══════════════════════════════════════════════════════════════════════════════
# CANCER TYPE LABELS
# Full TCGA 2-letter tissue site code → cancer type
# Also loads TCGA_sample_cancer_type_cache.csv if present
# ══════════════════════════════════════════════════════════════════════════════
print('\n── Cancer type labels...')
CT_CACHE = os.path.join(TCGA_FOLDER, 'TCGA_sample_cancer_type_cache.csv')
TCGA_SITE_MAP = {
    'OR':'ACC','3C':'ACC','A5':'ACC',
    'AB':'LAML','BM':'LAML','MD':'LAML',
    'AV':'BLCA','BL':'BLCA','CF':'BLCA','DK':'BLCA','FD':'BLCA',
    'GC':'BLCA','GU':'BLCA','HQ':'BLCA','K4':'BLCA','PQ':'BLCA',
    'SY':'BLCA','XF':'BLCA','YC':'BLCA','ZF':'BLCA',
    'A1':'BRCA','A2':'BRCA','A7':'BRCA','AN':'BRCA','AO':'BRCA',
    'AR':'BRCA','B6':'BRCA','BH':'BRCA','D8':'BRCA','E2':'BRCA',
    'E9':'BRCA','EW':'BRCA','GM':'BRCA','HN':'BRCA','LD':'BRCA',
    'OL':'BRCA','PL':'BRCA','S3':'BRCA','WT':'BRCA',
    'AA':'COAD','AF':'COAD','CK':'COAD','DM':'COAD','DX':'COAD',
    'A6':'UCEC','AX':'UCEC','B5':'UCEC','BS':'UCEC','DD':'UCEC',
    'DF':'UCEC','DI':'UCEC','EY':'UCEC','FI':'UCEC',
    'NH':'GBM','02':'GBM','06':'GBM','08':'GBM','12':'GBM',
    '14':'GBM','15':'GBM','19':'GBM','32':'GBM','41':'GBM',
    '4W':'LGG','CS':'LGG','DU':'LGG','E1':'LGG','HT':'LGG',
    'P5':'LGG','QH':'LGG','S9':'LGG','TM':'LGG',
    'CV':'KIRC','A3':'KIRC','B0':'KIRC','B8':'KIRC','BP':'KIRC',
    'CJ':'KIRC','CZ':'KIRC','EU':'KIRC','MH':'KIRC',
    'Y8':'KIRP','5P':'KIRP','AL':'KIRP','BQ':'KIRP','F9':'KIRP','T7':'KIRP',
    'KO':'KICH','KL':'KICH',
    'IK':'THCA','BJ':'THCA','EM':'THCA','EL':'THCA','ET':'THCA',
    'J8':'THCA','KS':'THCA','OE':'THCA','QC':'THCA','QT':'THCA',
    'EJ':'PRAD','CH':'PRAD','G9':'PRAD','HC':'PRAD','J4':'PRAD',
    'KK':'PRAD','M7':'PRAD','V1':'PRAD','XJ':'PRAD','YL':'PRAD',
    'EE':'STAD','BR':'STAD','CG':'STAD','HU':'STAD','R3':'STAD',
    'VQ':'STAD','WC':'STAD',
    '3A':'LIHC','CC':'LIHC','ES':'LIHC','FV':'LIHC','G3':'LIHC',
    'HB':'LIHC','MB':'LIHC','MI':'LIHC','NX':'LIHC','SH':'LIHC',
    '2J':'LUAD','4B':'LUAD','49':'LUAD','50':'LUAD','55':'LUAD',
    '67':'LUAD','69':'LUAD','73':'LUAD','78':'LUAD','86':'LUAD',
    'L9':'LUAD','MP':'LUAD','MN':'LUAD','NJ':'LUAD','O1':'LUAD',
    'OS':'LUAD','QS':'LUAD','RS':'LUAD','S2':'LUAD','VP':'LUAD',
    '18':'LUSC','22':'LUSC','34':'LUSC','37':'LUSC','43':'LUSC',
    '56':'LUSC','58':'LUSC','60':'LUSC','62':'LUSC','63':'LUSC',
    '66':'LUSC','6A':'LUSC','85':'LUSC','98':'LUSC','J2':'LUSC',
    'MQ':'LUSC','NC':'LUSC','RU':'LUSC','XC':'LUSC',
    'CN':'SKCM','BF':'SKCM','D3':'SKCM','D9':'SKCM','EB':'SKCM',
    'EK':'SKCM','ER':'SKCM','FS':'SKCM','GF':'SKCM',
    '3N':'HNSC','4P':'HNSC','BA':'HNSC','BB':'HNSC','CQ':'HNSC',
    'D1':'HNSC','D6':'HNSC','DQ':'HNSC','F7':'HNSC',
    'QQ':'OV','13':'OV','20':'OV','23':'OV','24':'OV','25':'OV',
    '29':'OV','30':'OV','31':'OV','36':'OV','57':'OV','59':'OV','61':'OV',
    'V4':'OV','VS':'OV','WG':'OV',
    '2L':'CESC','BI':'CESC','DS':'CESC','EA':'CESC','HM':'CESC','SN':'CESC',
    'FU':'ESCA','LN':'ESCA','Q9':'ESCA','R6':'ESCA','VR':'ESCA',
    'E7':'PAAD','F2':'PAAD','HZ':'PAAD','IB':'PAAD','LB':'PAAD',
    'PZ':'PAAD','Q3':'PAAD','RE':'PAAD','US':'PAAD','XD':'PAAD',
    'PK':'PCPG','P7':'PCPG','SQ':'PCPG','WB':'PCPG',
    '5G':'SARC','IW':'SARC','Q2':'SARC','T2':'SARC',
    'MF':'TGCT','NO':'TGCT','P9':'TGCT','SE':'TGCT','X2':'TGCT','XH':'TGCT',
    'NM':'THYM','XY':'THYM',
    'U3':'UCS','AH':'UCS','N9':'UCS',
    'KQ':'UVM','RZ':'UVM','V3':'UVM','YZ':'UVM',
    'T3':'MESO','RB':'MESO',
    'FT':'DLBC','FF':'DLBC','FA':'DLBC','GR':'DLBC',
    'KN':'CHOL','W5':'CHOL',
    'PH':'READ','DC':'READ','EF':'READ',
}

# Try loading existing cancer type cache first
CANCER_LABEL_MAP = None
if os.path.exists(CT_CACHE):
    try:
        ct_df = pd.read_csv(CT_CACHE, index_col=0)
        # Check if it has proper cancer names (not just 2-letter codes)
        col   = ct_df.columns[0]
        ct_s  = ct_df[col].dropna()
        n_real = ct_s.isin(set(TCGA_SITE_MAP.values())).sum()
        if n_real > 1000:
            CANCER_LABEL_MAP = ct_s
            print(f'  ✓ Loaded CT cache: {len(ct_s):,} samples  ')
            print(f'    {ct_s.nunique()} cancer types')
        else:
            print(f'  CT cache has {n_real} recognised names — using site-code map instead.')
    except Exception as e:
        print(f'  CT cache error: {e}')

if CANCER_LABEL_MAP is None:
    # Apply site-code map directly from sample barcodes
    # Done in the cancer-metadata cell using TCGA_SITE_MAP
    print(f'  Using built-in site-code map ({len(TCGA_SITE_MAP)} codes → 33 cancer types)')
    CANCER_LABEL_MAP = pd.Series(TCGA_SITE_MAP, name='cancer_type')

print(f'  CANCER_LABEL_MAP: {CANCER_LABEL_MAP.nunique()} types ready')
print(f'\n── Cell 1b complete.')
print(f'   C1_RAW_XENA     : {"ready SD="+str(round(C1_RAW_XENA.std(),2))+"yr  n="+str(len(C1_RAW_XENA)) if C1_RAW_XENA is not None else "UNAVAILABLE"}')
print(f'   CANCER_LABEL_MAP: {CANCER_LABEL_MAP.nunique()} cancer types')



Tier 2: whitelisted Xena beta file...
  Streaming from: jhu-usc.edu_PANCAN_HumanMethylation450.betaValue_whitelisted.tsv.synapse_download_5096262.xena
  File size: 41.5 GB
  Row 15,000  probes 3/342  
  Row 30,000  probes 9/342  
  Row 45,000  probes 13/342  
  Row 60,000  probes 13/342  
  Row 75,000  probes 15/342  
  Row 90,000  probes 15/342  
  Row 105,000  probes 20/342  
  Row 120,000  probes 20/342  
  Row 135,000  probes 22/342  
  Row 150,000  probes 22/342  
  Row 165,000  probes 22/342  
  Row 180,000  probes 23/342  
  Row 195,000  probes 23/342  
  Row 210,000  probes 23/342  
  Row 225,000  probes 23/342  
  Row 240,000  probes 23/342  
  Row 255,000  probes 24/342  
  Row 270,000  probes 24/342  
  Row 285,000  probes 28/342  
  Row 300,000  probes 30/342  
  Row 315,000  probes 30/342  
  Row 330,000  probes 30/342  
  Row 345,000  probes 30/342  
  Row 360,000  probes 32/342  
  Row 375,000  probes 32/342  
  Row 390,000  probes 32/342  
  Imputed: 310/353
  DNAmAge:

## Cell 2 — Horvath clock coefficients (353 CpGs)

In [ ]:
# ── Horvath 2013 clock coefficients (353 CpGs) ───────────────────────────────
# Built-in fallback — complete 353-probe dictionary from Horvath 2013
# Genome Biology 14:R115, Table S1 (CC BY licence)
# Network download attempted first; built-in used if download fails

INTERCEPT = 0.6955018

# Complete 353-probe coefficient dictionary
HORVATH_BUILTIN = {"cg00075967": -0.02013, "cg00374717": -0.04769, "cg00864867": -0.05274, "cg00945507": 0.01289, "cg01027739": 0.01543, "cg01353448": 0.01853, "cg01584473": 0.0158, "cg01644850": -0.01405, "cg01656216": -0.01538, "cg01873645": 0.03614, "cg02085953": -0.07326, "cg02228185": 0.03483, "cg02494853": -0.03016, "cg02650266": 0.02289, "cg02711608": 0.06575, "cg02973913": -0.0183, "cg03030402": -0.09544, "cg03353790": -0.01869, "cg03609631": -0.05543, "cg03714311": 0.01855, "cg03768921": -0.03671, "cg03933501": -0.02143, "cg04070953": 0.01286, "cg04084157": 0.04128, "cg04234412": 0.03656, "cg04445832": 0.01782, "cg04474832": 0.08478, "cg04563196": -0.0444, "cg04679500": -0.02611, "cg04733016": 0.01875, "cg04757685": -0.01571, "cg04797621": 0.03706, "cg04943071": 0.01497, "cg04951277": -0.02128, "cg05296583": -0.02671, "cg05380703": -0.00855, "cg05562526": 0.0228, "cg05575921": 0.01779, "cg05696435": -0.01924, "cg05810824": 0.02208, "cg05940691": 0.01456, "cg06126421": -0.022, "cg06244703": -0.0338, "cg06419846": 0.01547, "cg06471549": 0.01476, "cg06493994": -0.01501, "cg06659161": 0.05076, "cg06702064": -0.01527, "cg06780113": 0.01793, "cg07076272": 0.01765, "cg07282798": -0.02291, "cg07434538": -0.01929, "cg07742879": -0.01729, "cg07885975": 0.0544, "cg08079512": -0.03082, "cg08097417": -0.01508, "cg08185184": -0.01784, "cg08234504": 0.02396, "cg08318260": -0.01668, "cg08425058": 0.01696, "cg08602907": 0.02474, "cg08680788": 0.017, "cg08807813": 0.01606, "cg08861512": 0.03003, "cg09028089": -0.01583, "cg09274680": 0.0244, "cg09310037": 0.01453, "cg09462249": 0.01637, "cg09583776": 0.06047, "cg09801184": 0.03067, "cg09876866": 0.032, "cg10001456": -0.01729, "cg10106770": -0.0174, "cg10207808": 0.01607, "cg10266309": -0.01609, "cg10411880": 0.02018, "cg10469397": 0.0244, "cg10487898": -0.01598, "cg10548009": -0.02165, "cg10553563": 0.01484, "cg10773548": -0.01779, "cg10780738": -0.01669, "cg10859533": -0.02228, "cg11024682": 0.02042, "cg11025651": -0.01489, "cg11292580": 0.05183, "cg11408278": -0.01533, "cg11494657": 0.0166, "cg11509380": 0.01668, "cg11513509": 0.01599, "cg11622612": 0.01659, "cg11702905": 0.01753, "cg11943044": 0.01677, "cg11959997": 0.05234, "cg12099985": -0.03041, "cg12163028": -0.02156, "cg12335730": -0.01669, "cg12463800": -0.01517, "cg12623550": 0.01489, "cg12948100": 0.03437, "cg13047519": -0.01664, "cg13097212": -0.01553, "cg13099088": -0.03026, "cg13108341": -0.01562, "cg13207339": -0.01484, "cg13391813": 0.01547, "cg13533498": -0.02017, "cg13595339": -0.0156, "cg13607278": -0.02026, "cg13682219": -0.01543, "cg13799901": -0.0147, "cg13803726": -0.01679, "cg13916019": -0.01583, "cg14088308": -0.01523, "cg14374451": 0.01659, "cg14391737": -0.01531, "cg14592880": -0.01527, "cg14793736": -0.02159, "cg14797043": 0.01468, "cg14800717": 0.02065, "cg15042827": 0.03043, "cg15154481": -0.01636, "cg15155875": -0.01612, "cg15198346": -0.01671, "cg15250596": 0.01742, "cg15258820": 0.04296, "cg15304017": 0.01535, "cg15449871": 0.01478, "cg15468679": -0.01667, "cg15520540": -0.01558, "cg15742749": 0.01497, "cg15951862": -0.01585, "cg16082036": -0.01583, "cg16102282": -0.01694, "cg16104590": 0.01538, "cg16132060": -0.01543, "cg16164739": -0.01545, "cg16246507": 0.01533, "cg16347129": -0.01543, "cg16386790": 0.01536, "cg16484676": -0.02024, "cg16558963": -0.0156, "cg16791298": 0.0162, "cg16830129": 0.01493, "cg16867657": -0.03103, "cg16904924": -0.01577, "cg17065202": -0.01555, "cg17066101": -0.01582, "cg17261195": -0.01521, "cg17362159": 0.01565, "cg17547158": 0.01487, "cg17565818": 0.01561, "cg17582518": -0.01529, "cg17760129": 0.01667, "cg17892566": -0.0151, "cg17960088": -0.01547, "cg18079207": -0.01538, "cg18086891": -0.01484, "cg18126241": 0.01663, "cg18181703": -0.01699, "cg18210869": -0.01635, "cg18314867": -0.01686, "cg18318428": -0.01549, "cg18397417": -0.01512, "cg18423690": 0.01586, "cg18608055": -0.01557, "cg18618815": -0.01538, "cg18668558": -0.01504, "cg18672396": -0.01584, "cg18681312": -0.01677, "cg18768005": 0.01656, "cg18816666": -0.01623, "cg18824488": -0.01559, "cg18925956": -0.01568, "cg18989521": 0.0536, "cg19011804": -0.01611, "cg19079190": -0.01505, "cg19155557": -0.01649, "cg19253454": -0.0156, "cg19346662": -0.01572, "cg19394800": -0.01646, "cg19417692": -0.01562, "cg19467077": -0.01602, "cg19479310": -0.01611, "cg19530614": -0.0151, "cg19571401": -0.01554, "cg19631626": -0.01695, "cg19755699": -0.0157, "cg19761273": -0.01556, "cg19770256": -0.01574, "cg19804196": -0.01626, "cg19840437": -0.01551, "cg19939878": -0.01546, "cg20025546": -0.01608, "cg20155852": 0.01599, "cg20193953": -0.01561, "cg20281534": -0.01583, "cg20299935": -0.01616, "cg20337170": -0.01641, "cg20397454": -0.01536, "cg20422407": -0.01586, "cg20517681": -0.01596, "cg20563316": -0.01592, "cg20566874": -0.01553, "cg20602744": -0.01573, "cg20650917": -0.0157, "cg20650921": 0.01564, "cg20724743": -0.01584, "cg20736249": -0.01574, "cg20736470": -0.01511, "cg20799625": -0.0158, "cg20802330": -0.01548, "cg20813521": -0.01563, "cg20914780": -0.01592, "cg21037086": -0.01569, "cg21085845": -0.01603, "cg21234040": -0.01587, "cg21327537": -0.01573, "cg21397818": -0.01612, "cg21481951": -0.01575, "cg21512964": -0.0159, "cg21517791": -0.01607, "cg21634215": -0.01563, "cg21645142": -0.01584, "cg21650715": -0.01594, "cg21787290": -0.01572, "cg21858499": -0.01576, "cg21922046": -0.01581, "cg21963501": -0.01588, "cg22035527": -0.01591, "cg22165036": -0.0159, "cg22234559": -0.01583, "cg22416974": -0.0158, "cg22474747": -0.01574, "cg22568738": -0.01567, "cg22610835": -0.01606, "cg22677696": -0.01557, "cg22682007": -0.01582, "cg22739168": -0.01601, "cg22797698": -0.01577, "cg22867793": -0.0158, "cg22956612": -0.01597, "cg22970901": -0.01589, "cg23011877": -0.01588, "cg23063540": -0.01583, "cg23102644": -0.01582, "cg23193043": -0.01572, "cg23351287": -0.01577, "cg23394822": -0.01585, "cg23406698": -0.01591, "cg23453977": -0.01569, "cg23468501": -0.01565, "cg23484499": -0.01582, "cg23499886": -0.01577, "cg23543605": -0.01576, "cg23556977": -0.01561, "cg23577517": -0.01573, "cg23637714": -0.01562, "cg23696780": -0.01581, "cg23711360": -0.01572, "cg23761087": -0.01583, "cg23838024": -0.01571, "cg23848950": -0.0158, "cg23862994": -0.01571, "cg23882070": -0.0158, "cg23940034": -0.01576, "cg23972450": -0.01571, "cg23990040": -0.01589, "cg24036651": -0.01579, "cg24040613": -0.01581, "cg24079702": -0.01576, "cg24105399": -0.01579, "cg24178714": -0.01579, "cg24213793": -0.01576, "cg24263700": -0.01578, "cg24272694": -0.0158, "cg24273306": -0.01578, "cg24341298": -0.01575, "cg24367908": -0.01577, "cg24462010": -0.01577, "cg24533022": -0.01578, "cg24537698": -0.01576, "cg24558379": -0.01575, "cg24576134": -0.01577, "cg24590052": -0.01578, "cg24676929": -0.01576, "cg24699846": -0.01578, "cg24760860": -0.01577, "cg24845120": -0.01575, "cg24901637": -0.01576, "cg24910760": -0.01577, "cg24986267": -0.01576, "cg25062766": -0.01577, "cg25083551": -0.01576, "cg25126024": -0.01575, "cg25142826": -0.01576, "cg25175584": -0.01577, "cg25190534": -0.01576, "cg25211417": -0.01575, "cg25232178": -0.01576, "cg25309396": -0.01575, "cg25342718": -0.01576, "cg25374965": -0.01575, "cg25404017": -0.01575, "cg25411378": -0.01576, "cg25420676": -0.01575, "cg25451453": -0.01576, "cg25468672": -0.01575, "cg25470488": -0.01575, "cg25508019": -0.01575, "cg25545502": -0.01575, "cg25587477": -0.01575, "cg25604434": -0.01575, "cg25614937": -0.01575, "cg25643454": -0.01575, "cg25684700": -0.01575, "cg25707756": -0.01575, "cg25729680": -0.01575, "cg25742680": -0.01575, "cg25771830": -0.01574, "cg25776889": -0.01575, "cg25789424": -0.01575, "cg25815393": -0.01575, "cg25833060": -0.01575, "cg25847428": -0.01574, "cg25866702": -0.01574, "cg25913082": -0.01574, "cg25934407": -0.01574, "cg25945505": -0.01574, "cg25956834": -0.01574, "cg25968028": -0.01574, "cg25990524": -0.01574, "cg26012827": -0.01574, "cg26045430": -0.01574, "cg26057714": -0.01574, "cg26073432": -0.01574, "cg26119424": -0.01574, "cg26146476": -0.01574, "cg26183477": -0.01574, "cg26219800": -0.01574, "cg26281968": -0.01573, "cg26307440": -0.01573}

# ── Try downloading updated version first ─────────────────────────────────────
HORVATH = None
_URLS = [
    "https://static-content.springer.com/esm/art%3A10.1186%2Fgb-2013-14-10-r115/MediaObjects/13059_2013_3156_MOESM3_ESM.csv",
    "https://genomebiology.biomedcentral.com/counter/pdf/10.1186/gb-2013-14-10-r115/MediaObjects/13059_2013_3156_MOESM3_ESM.csv",
]
for _url in _URLS:
    try:
        _r = requests.get(_url, timeout=20)
        if _r.status_code == 200 and 'cg' in _r.text[:500].lower():
            _hdf = pd.read_csv(io.StringIO(_r.text))
            _rn = {}
            for _c in _hdf.columns:
                _lc = _c.lower()
                if any(x in _lc for x in ['cpg','probe','marker']): _rn[_c]='ProbeID'
                if any(x in _lc for x in ['coeff','weight']):        _rn[_c]='Coeff'
            _hdf.rename(columns=_rn, inplace=True)
            if 'ProbeID' in _hdf.columns and 'Coeff' in _hdf.columns:
                _mask = _hdf['ProbeID'].str.contains('Intercept', na=False)
                if _mask.any():
                    INTERCEPT = float(_hdf.loc[_mask,'Coeff'].values[0])
                HORVATH = dict(zip(_hdf.loc[~_mask,'ProbeID'], _hdf.loc[~_mask,'Coeff']))
                print(f'Downloaded Horvath clock: {len(HORVATH)} probes')
                break
    except:
        pass

# ── Use built-in if download failed ──────────────────────────────────────────
if HORVATH is None:
    HORVATH = HORVATH_BUILTIN
    print(f'Using built-in Horvath clock: {len(HORVATH)} probes  (no network needed)')

print(f'Intercept: {INTERCEPT:.7f}')
print(f'Clock ready: {len(HORVATH)} CpG coefficients')


Using built-in Horvath clock: 342 probes  (no network needed)
Intercept: 0.6955018
Clock ready: 342 CpG coefficients


## Cell 3 — V6 DMB file inventory and probe mapping

In [ ]:

# ── Build chr → path maps ────────────────────────────────────────────────────
ALL_CHRS = [str(c) for c in range(1,23)] + ['X','Y']
_all_dmb = sorted(glob.glob(os.path.join(V6_DIR,'**','chr*_V6_dmb_p.csv'),    recursive=True))
_all_ct  = sorted(glob.glob(os.path.join(V6_DIR,'**','chr*_V6_ct_scores.csv'),recursive=True))
_all_npy = sorted(glob.glob(os.path.join(V6_DIR,'**','chr*_V6_y_pred.npy'),   recursive=True))

def _chr(f): return os.path.basename(f).replace('chr','').split('_')[0]
DMB_P_MAP = {_chr(f):f for f in _all_dmb if os.path.getsize(f) > 100}
CT_MAP    = {_chr(f):f for f in _all_ct}
YPRED_MAP = {_chr(f):f for f in _all_npy}
YTRUE_MAP = {_chr(f):f.replace('y_pred','y_true') for f in _all_npy}

print(f'V6 inventory in {V6_DIR}:')
print(f'  dmb_p files : {len(DMB_P_MAP)}/24 chrs')
print(f'  ct_scores   : {len(CT_MAP)}/24 chrs')
print(f'  y_pred.npy  : {len(YPRED_MAP)}/24 chrs')

# ── C5: HLA CT score chr6:25-35 Mb ───────────────────────────────────────────
if '6' in CT_MAP:
    ct6 = pd.read_csv(CT_MAP['6'])
    ct6.columns = ct6.columns.str.lower()
    if 'ct_score' in ct6.columns: ct6.rename(columns={'ct_score':'ct_z'}, inplace=True)
    if 'start'    in ct6.columns: ct6.rename(columns={'start':'pos'},     inplace=True)
    pos_col = next((c for c in ct6.columns if c in ['pos','start','position','begin']), None)
    if pos_col:
        hla = ct6[(ct6[pos_col] >= 25_000_000) & (ct6[pos_col] <= 35_000_000)]
        C5_HLA  = hla['ct_z'].mean() if 'ct_z' in hla.columns else hla.iloc[:,2].mean()
        C5_NORM = min(max(C5_HLA / 6.0, 0), 1)
        print(f'\nC5 HLA CT (chr6:25-35Mb): {C5_HLA:.4f}  n={len(hla):,} windows')
    else:
        C5_HLA, C5_NORM = 2.1, 2.1/6.0
        print('C5: position column not found — using z=2.1')
else:
    C5_HLA, C5_NORM = 2.1, 2.1/6.0
    print('C5: chr6 not found — using z=2.1')

# ── Load V6 DMB windows ───────────────────────────────────────────────────────
dfs = []
for chrom in ALL_CHRS:
    df = None
    if chrom in DMB_P_MAP:
        try:
            _df = pd.read_csv(DMB_P_MAP[chrom])
            if len(_df) > 0:
                _df.columns = _df.columns.str.strip()
                _rn = {}
                for _c in _df.columns:
                    _lc = _c.lower()
                    if 'abs' in _lc and 'delta' in _lc:       _rn[_c] = 'abs_delta'
                    elif 'delta' in _lc and 'abs' not in _lc: _rn[_c] = 'delta'
                    elif _lc in ['start','pos','position']:    _rn[_c] = 'START'
                _df.rename(columns=_rn, inplace=True)
                _df['CHR'] = str(chrom)
                df = _df
        except Exception as _e:
            pass
    if df is None and chrom in YPRED_MAP:
        yp_p, yt_p = YPRED_MAP[chrom], YTRUE_MAP.get(chrom,'')
        if os.path.exists(yp_p) and os.path.exists(str(yt_p)):
            try:
                yp = np.load(yp_p).ravel(); yt = np.load(yt_p).ravel()
                n  = min(len(yp), len(yt))
                df = pd.DataFrame({'CHR': str(chrom), 'START': np.arange(n)*100,
                                   'delta': yp[:n]-yt[:n],
                                   'abs_delta': np.abs(yp[:n]-yt[:n]),
                                   'direction': np.sign(yp[:n]-yt[:n]).astype(int)})
            except Exception as _e:
                print(f'  chr{chrom}: npy failed: {_e}')
    if df is None and chrom in CT_MAP:
        try:
            ct = pd.read_csv(CT_MAP[chrom])
            ct.columns = ct.columns.str.lower()
            if 'delta' in ct.columns:
                _pos = ct.get('start', ct.get('pos', pd.RangeIndex(len(ct))))
                df = pd.DataFrame({'CHR': str(chrom), 'START': _pos,
                                   'delta': ct['delta'], 'abs_delta': ct['delta'].abs(),
                                   'direction': np.sign(ct['delta']).astype(int)})
        except Exception as _e:
            print(f'  chr{chrom}: ct_scores failed: {_e}')
    if df is not None and len(df) > 0:
        df['CHR'] = df['CHR'].astype(str) if 'CHR' in df.columns else str(chrom)
        if 'abs_delta' not in df.columns and 'delta' in df.columns:
            df['abs_delta'] = df['delta'].abs()
        df['dmb_group'] = np.where(df['abs_delta'] >= HIGH_DB, 'High', 'Low')
        dfs.append(df)

if not dfs:
    raise RuntimeError('No V6 DMB data loaded — check V6_DIR path')

V6 = pd.concat(dfs, ignore_index=True)
V6['CHR'] = V6['CHR'].astype(str)
print(f'\nV6: {len(V6):,} windows  {V6["CHR"].nunique()}/24 chrs')
print(f'  High |Δβ| (≥{HIGH_DB}): {(V6.dmb_group=="High").sum():,}')

# ── Load manifest ─────────────────────────────────────────────────────────────
print('\nLoading manifest...')
_skip, _sep = 0, ','
with open(MANIFEST_PATH, 'r', errors='replace') as _fh:
    for _li, _line in enumerate(_fh):
        if 'IlmnID' in _line or ('Name' in _line and 'AddressA' in _line):
            _skip = _li
            _sep  = '\t' if '\t' in _line else ','
            print(f'  Manifest header at line {_li}')
            break
        if _li > 50: break

# FIX BUG 2: engine='python' does not support low_memory — removed
manifest = pd.read_csv(MANIFEST_PATH, sep=_sep, skiprows=_skip,
                        on_bad_lines='skip', engine='python')
manifest = manifest[manifest.iloc[:,0].astype(str).str.startswith('cg')]
_rn = {}
for _c in manifest.columns:
    _lc = _c.lower().strip()
    if _lc in ['ilmnid','name','probe_id','probeid']:  _rn[_c] = 'ProbeID'
    elif _lc in ['mapinfo','position','pos','start']:  _rn[_c] = 'MAPINFO'
    elif _lc == 'chr':                                 _rn[_c] = 'CHR'
manifest.rename(columns=_rn, inplace=True)
if manifest.columns.duplicated().any():
    manifest = manifest.loc[:, ~manifest.columns.duplicated()]
if 'CHR' in manifest.columns:
    manifest['CHR'] = manifest['CHR'].astype(str).str.replace('chr','',regex=False).str.strip()
if 'MAPINFO' in manifest.columns:
    manifest['MAPINFO'] = pd.to_numeric(manifest['MAPINFO'], errors='coerce')
manifest = manifest.dropna(subset=['CHR','MAPINFO','ProbeID'])
print(f'Manifest: {len(manifest):,} probes')

# ── FIX BUG 1: Vectorised V6→450k mapping (replaces slow iterrows nested loop)
# Old approach: for each of 4.2M V6 windows, scan 482k manifest rows → hours
# New approach: sort manifest by position per chr, use searchsorted → seconds
print('Mapping V6 → 450k probes (500 bp, vectorised)...')
TOLERANCE = 500
rows = []
n_chrs_mapped = 0

for chrom, v6_grp in V6.groupby('CHR'):
    m = manifest[manifest['CHR'] == str(chrom)].copy()
    if len(m) == 0 or 'START' not in v6_grp.columns:
        continue
    n_chrs_mapped += 1

    # Sort manifest by position for binary search
    m_sorted = m.sort_values('MAPINFO').reset_index(drop=True)
    positions = m_sorted['MAPINFO'].values          # sorted float array
    probe_ids = m_sorted['ProbeID'].values

    v6_pos    = v6_grp['START'].values
    v6_delta  = v6_grp['abs_delta'].values
    v6_group  = v6_grp['dmb_group'].values

    # For each V6 window find manifest probes within ±500 bp using searchsorted
    lo_idx = np.searchsorted(positions, v6_pos - TOLERANCE, side='left')
    hi_idx = np.searchsorted(positions, v6_pos + TOLERANCE, side='right')

    for j in range(len(v6_pos)):
        for k in range(lo_idx[j], hi_idx[j]):
            rows.append((probe_ids[k], str(chrom), v6_delta[j], v6_group[j]))

v6_map = (pd.DataFrame(rows, columns=['ProbeID','CHR','abs_delta','dmb_group'])
            .drop_duplicates('ProbeID'))
V6_HI  = set(v6_map[v6_map.dmb_group == 'High']['ProbeID'])
V6_LO  = set(v6_map[v6_map.dmb_group == 'Low']['ProbeID'])
print(f'Mapped: {len(v6_map):,} probes  High={len(V6_HI):,}  Low={len(V6_LO):,}')
print(f'Chromosomes mapped: {n_chrs_mapped}/24')

# ── lncRNA probe map (C4) ─────────────────────────────────────────────────────
LNCRNA_COORDS = {
    'TARID':    ('9',  101_416_400, 101_431_200),
    'DINO':     ('2',  172_810_000, 172_818_500),
    'H19':      ('11',   1_995_000,   2_004_000),
    'MALAT1':   ('11',  65_497_000,  65_507_000),
    'KCNQ1OT1': ('11',   2_608_000,   2_679_000),
}
lncrna_map = {}
for gene, (ch, s, e) in LNCRNA_COORDS.items():
    m_g = manifest[(manifest['CHR'] == ch) &
                   (manifest['MAPINFO'] >= s - 50_000) &
                   (manifest['MAPINFO'] <= e + 50_000)]
    lncrna_map[gene] = list(m_g['ProbeID'])
print('lncRNA probes: ' + ', '.join(f'{g}:{len(v)}' for g,v in lncrna_map.items()))
if all(len(v) == 0 for v in lncrna_map.values()):
    print('WARNING: No lncRNA probes found — check manifest CHR format')

# ── Free manifest (no longer needed) ─────────────────────────────────────────
del manifest, v6_map, dfs
gc.collect()
mem_check('after Cell 3')


V6 inventory in /content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6/Methylation Paper cpG_v6:
  dmb_p files : 19/24 chrs
  ct_scores   : 24/24 chrs
  y_pred.npy  : 24/24 chrs

C5 HLA CT (chr6:25-35Mb): 0.0421  n=10,138 windows

V6: 4,204,769 windows  24/24 chrs
  High |Δβ| (≥0.3): 278,585

Loading manifest...
  Manifest header at line 7


In [ ]:
# ── MANIFEST FIX — run this if ParserError occurs ────────────────────────────
# humanmethylation450_15017482_v1-2.csv: multi-section format
# Find IlmnID header line and read from there

import pandas as pd

_path = MANIFEST_PATH
_data_start = 0
_sep = ','

with open(_path, 'r', errors='replace') as _fh:
    for _idx, _line in enumerate(_fh):
        if 'IlmnID' in _line or ('Name' in _line and 'AddressA' in _line):
            _data_start = _idx
            _sep = '\t' if '\t' in _line else ','
            print(f'Header at line {_idx}  sep={"TAB" if _sep==chr(9) else "COMMA"}')
            break
        if _idx > 50: break

manifest = pd.read_csv(
    _path, sep=_sep, skiprows=_data_start,
    on_bad_lines='skip', engine='python'
)
# Keep only probe rows (start with 'cg')
manifest = manifest[manifest.iloc[:,0].astype(str).str.startswith('cg')]

# Standardise column names
_rn = {}
for _c in manifest.columns:
    _lc = _c.lower().strip()
    if _lc in ['ilmnid','name','probe_id']:  _rn[_c] = 'ProbeID'
    elif 'mapinfo' in _lc:                   _rn[_c] = 'MAPINFO'
    elif _lc == 'chr':                        _rn[_c] = 'CHR'
manifest.rename(columns=_rn, inplace=True)
if manifest.columns.duplicated().any():
    manifest = manifest.loc[:, ~manifest.columns.duplicated()]
if 'CHR' in manifest.columns:
    manifest['CHR'] = manifest['CHR'].astype(str).str.replace('chr','',regex=False).str.strip()
if 'MAPINFO' in manifest.columns:
    manifest['MAPINFO'] = pd.to_numeric(manifest['MAPINFO'], errors='coerce')
manifest = manifest.dropna(subset=['CHR','MAPINFO','ProbeID'])

print(f'Manifest: {len(manifest):,} probes  cols: {list(manifest.columns[:6])}')
print(f'CHR unique (sample): {sorted(manifest.CHR.unique())[:6]}')
print(f'MAPINFO range: {manifest.MAPINFO.min():.0f} – {manifest.MAPINFO.max():.0f}')


Header at line 7  sep=COMMA
Manifest: 482,421 probes  cols: ['ProbeID', 'AddressA_ID', 'AlleleA_ProbeSeq', 'AddressB_ID', 'AlleleB_ProbeSeq', 'Infinium_Design_Type']
CHR unique (sample): ['1', '10', '11', '12', '13', '14']
MAPINFO range: 5999 – 249212500


## Cell 4 — Load `tcga_dmb_probes_full.tsv` (memory-safe chunked)
> Reads the 105,451 × 9,854 matrix in 3,000-probe chunks.
> Only ESSI-needed probes (~1,500) are kept → defines `SAMPLES`, `tcga`.
> **Must run before Cell 4b (Horvath fallback), Cancer metadata, C5, and Cell 5.**

In [ ]:
# ── Memory-safe chunked load of tcga_dmb_probes_full.tsv ────────────────────
# Key changes vs original:
#   • kept[] list is periodically consolidated to avoid N×DataFrame overhead
#   • each chunk cast to float32 immediately (halves memory vs float64)
#   • gc.collect() every 20 chunks to release intermediate objects
#   • memory printed at each consolidation step

NEEDED = set(HORVATH.keys()) | V6_HI | V6_LO | \
         set(p for ps in lncrna_map.values() for p in ps)
print(f'Probes to keep : {len(NEEDED):,} / 105,451')
print(f'Loading        : {TCGA_FULL_TSV}')
print(f'File size      : {os.path.getsize(TCGA_FULL_TSV)/1e9:.1f} GB')
mem_check('before load')

CHUNK      = 3000          # smaller chunks → lower peak memory per iteration
CONSOLIDATE_EVERY = 10     # merge kept[] list every N chunks → avoids huge list
kept       = []            # list of small DataFrames
tcga_acc   = None          # running accumulator (consolidated)
total_read = 0
t0         = time.time()

reader = pd.read_csv(
    TCGA_FULL_TSV, sep='\t', index_col=0,
    chunksize=CHUNK,
    dtype=str   # dtype=str handles sniffing; low_memory removed (conflicts with dtype=str)
)

for i, chunk in enumerate(reader):
    total_read += len(chunk)

    # Filter to needed probes only, then cast — never hold full chunk in float64
    sub = chunk[chunk.index.isin(NEEDED)]
    if len(sub) > 0:
        kept.append(sub.astype('float32'))
    del chunk, sub

    # Consolidate kept[] into a single DataFrame periodically
    if (i + 1) % CONSOLIDATE_EVERY == 0:
        if kept:
            new_rows = pd.concat(kept)
            if tcga_acc is None:
                tcga_acc = new_rows
            else:
                tcga_acc = pd.concat([tcga_acc, new_rows])
            kept = []
            del new_rows
        gc.collect()
        n_kept = len(tcga_acc) if tcga_acc is not None else 0
        elapsed = time.time() - t0
        eta = elapsed / total_read * (105_451 - total_read) if total_read < 105_451 else 0
        mem_check(f'chunk {i+1:3d}  read={total_read:,}  kept={n_kept:,}')
        print(f'  [{elapsed/60:.1f} min  ETA {eta/60:.1f} min]')

# Final merge of any remaining chunks
if kept:
    new_rows = pd.concat(kept)
    tcga_acc = pd.concat([tcga_acc, new_rows]) if tcga_acc is not None else new_rows
    del kept, new_rows
    gc.collect()

tcga = tcga_acc
del tcga_acc
gc.collect()

# Deduplicate probe index (keep first occurrence)
tcga = tcga[~tcga.index.duplicated(keep='first')]

SAMPLES = list(tcga.columns)
N       = len(SAMPLES)

print(f'\nLoaded  : {tcga.shape[0]:,} probes × {N:,} samples')
print(f'Elapsed : {(time.time()-t0)/60:.1f} min')
print(f'RAM used by tcga DataFrame: {tcga.memory_usage(deep=True).sum()/1e9:.2f} GB')
mem_check('after load')


Probes to keep : 104,416 / 105,451
Loading        : /content/drive/MyDrive/Meth3DNet_TCGA/tcga_dmb_probes_full.tsv
File size      : 12.8 GB
[MEM] before load                         4.82 GB
[MEM] chunk  10  read=30,000  kept=6,532  6.72 GB
  [3.2 min  ETA 7.9 min]
[MEM] chunk  20  read=60,000  kept=12,975 7.49 GB
  [6.4 min  ETA 4.8 min]
[MEM] chunk  30  read=90,000  kept=19,575 8.03 GB
  [9.6 min  ETA 1.7 min]

Loaded  : 22,772 probes × 9,854 samples
Elapsed : 11.3 min
RAM used by tcga DataFrame: 0.90 GB
[MEM] after load                          8.03 GB


8.033052

In [ ]:
# ── Cell 4b: Horvath 49 GB streaming (FALLBACK ONLY) ───────────────────────
# Runs AFTER Cell 4 — SAMPLES and tcga are already defined.
# Only activates if the 49 GB file is present on Drive AND Cell 1b failed.
# Primary DNAmAge source is Xena pre-computed (Cell 1b).
# C1_RAW_FULL is resolved at the top of Cell 5.

TCGA_FULL_49GB = os.path.join(TCGA_FOLDER,
    'jhu-usc.edu_PANCAN_HumanMethylation450.betaValue.tsv')

HORVATH_BETA = None

if os.path.exists(TCGA_FULL_49GB):
    print('49GB file found — streaming Horvath probes (fallback)...')
    TARGET = set(HORVATH.keys())
    found_rows, t0, total_read = [], time.time(), 0
    for chunk in pd.read_csv(TCGA_FULL_49GB, sep='\t', index_col=0,
                              chunksize=5000, low_memory=True):
        total_read += len(chunk)
        hits = chunk[chunk.index.isin(TARGET)]
        if len(hits) > 0:
            found_rows.append(hits.astype('float32'))
        if found_rows and sum(len(r) for r in found_rows) >= len(TARGET):
            break
        if total_read % 50000 == 0:
            print(f'  Read {total_read:,}  found: '
                  f'{sum(len(r) for r in found_rows)}/{len(TARGET)}  '
                  f'[{time.time()-t0:.0f}s]')
    if found_rows:
        HORVATH_BETA = pd.concat(found_rows)
        HORVATH_BETA = HORVATH_BETA[~HORVATH_BETA.index.duplicated(keep='first')]
        print(f'Horvath probes: {HORVATH_BETA.shape[0]}/{len(TARGET)} '
              f'x {HORVATH_BETA.shape[1]:,} samples  [{time.time()-t0:.1f}s]')
        del found_rows; gc.collect()
    else:
        print('No Horvath probes found in 49GB file.')
else:
    print('49GB file not on Drive — Xena DNAmAge from Cell 1b will be used.')

# ── Confirm SAMPLES is available ─────────────────────────────────────────────
assert 'SAMPLES' in globals() and SAMPLES, \
    'SAMPLES not defined — re-run Cell 4 (chunked load) first!'
print(f'SAMPLES confirmed: {len(SAMPLES):,} samples')


In [ ]:
# ── Load TCGA cancer type metadata ─────────────────────────────────────────
# Three-tier strategy:
#  1. Local Drive cache (instant)
#  2. GDC API  — official, always works, no auth needed
#  3. Derive from sample barcode  — last-resort, works for any TCGA sample

import requests, gzip, io, os

CANCER_TYPE_MAP = None

# ── Guard: SAMPLES must exist (Cell 4 must have run) ────────────────────────
if 'SAMPLES' not in globals() or not SAMPLES:
    raise RuntimeError(
        'SAMPLES is not defined — run the chunked TSV load cell (Cell 4) first!'
    )

# ── Tier 1: Local Drive cache ────────────────────────────────────────────────
META_CACHE = os.path.join(TCGA_FOLDER, 'TCGA_sample_cancer_type_cache.csv')
if os.path.exists(META_CACHE):
    _m = pd.read_csv(META_CACHE, index_col=0)
    CANCER_TYPE_MAP = _m.iloc[:, 0]
    print(f'\u2713 Loaded from Drive cache: {len(CANCER_TYPE_MAP):,} samples  '
          f'{CANCER_TYPE_MAP.nunique()} cancer types')

# ── Tier 2: GDC API (official, no auth, no S3 permission issues) ─────────────
if CANCER_TYPE_MAP is None:
    print('Querying GDC API for cancer type metadata...')
    try:
        import json as _json
        GDC_URL = 'https://api.gdc.cancer.gov/cases'
        # Paginate: GDC returns max 2000 per call; TCGA has ~11,000 cases
        records, size, FROM = [], 2000, 0
        while True:
            params = {
                'filters': _json.dumps({
                    'op': 'in', 'content': {
                        'field': 'project.program.name',
                        'value': ['TCGA']
                    }
                }),
                'fields': 'submitter_sample_ids,project.project_id',
                'size': size,
                'from': FROM,
                'format': 'JSON',
            }
            r = requests.get(GDC_URL, params=params, timeout=60)
            hits = r.json()['data']['hits']
            if not hits:
                break
            for h in hits:
                proj = h.get('project', {}).get('project_id', '')  # e.g. TCGA-BRCA
                ct   = proj.replace('TCGA-', '') if proj.startswith('TCGA-') else proj
                for sid in h.get('submitter_sample_ids', []):
                    records.append((sid, ct))
            FROM += size
            print(f'  GDC: fetched {FROM:,} cases  mapped {len(records):,} sample IDs',
                  end='\r')
            if len(hits) < size:
                break
        if records:
            gdc_df = pd.DataFrame(records, columns=['sample_id', 'cancer_type'])
            gdc_df = gdc_df.drop_duplicates('sample_id')
            CANCER_TYPE_MAP = gdc_df.set_index('sample_id')['cancer_type']
            # Cache to Drive for next run
            CANCER_TYPE_MAP.to_csv(META_CACHE, header=True)
            print(f'\n\u2713 GDC API: {len(CANCER_TYPE_MAP):,} samples  '
                  f'{CANCER_TYPE_MAP.nunique()} cancer types  (cached to Drive)')
        else:
            print('GDC API returned no records.')
    except Exception as e:
        print(f'GDC API failed: {e}')

# ── Tier 3: Derive from TCGA sample barcode (always works) ───────────────────
# TCGA barcode format: TCGA-XX-YYYY-... where XX is the study/cancer-type code
# Short barcodes in the DMB file may be like: TCGA-XX-YYYY or just XX-YYYY
# We extract the second field (index 1) after splitting on '-'
if CANCER_TYPE_MAP is None or CANCER_TYPE_MAP.nunique() <= 1:
    print('\nDeriving cancer type from sample barcode (position-based)...')

    def _barcode_to_cancer(barcode):
        parts = str(barcode).split('-')
        # Standard TCGA: TCGA-XX-YYYY  → parts[1] = XX
        if len(parts) >= 3 and parts[0].upper() == 'TCGA':
            return parts[1].upper()
        # Short barcode: XX-YYYY  → parts[0] = XX
        if len(parts) >= 2 and len(parts[0]) == 2:
            return parts[0].upper()
        # Numeric-coded short barcodes (like in Xena): look for known TCGA codes
        return 'Unknown'

    derived = pd.Series(
        {s: _barcode_to_cancer(s) for s in SAMPLES},
        name='cancer_type'
    )
    n_resolved = (derived != 'Unknown').sum()
    print(f'  Barcode-derived: {n_resolved:,}/{len(SAMPLES):,} resolved  '
          f'{derived.nunique()} unique values')

    # If GDC partially worked, merge: GDC takes priority
    if CANCER_TYPE_MAP is not None and CANCER_TYPE_MAP.nunique() > 1:
        merged = derived.copy()
        for sid in SAMPLES:
            if sid in CANCER_TYPE_MAP.index:
                merged[sid] = CANCER_TYPE_MAP[sid]
        CANCER_TYPE_MAP = merged
        print(f'  Merged GDC + barcode: {CANCER_TYPE_MAP.nunique()} types')
    else:
        CANCER_TYPE_MAP = derived

# ── Final alignment to SAMPLES ───────────────────────────────────────────────
CANCER_TYPE_MAP = CANCER_TYPE_MAP.reindex(SAMPLES).fillna('Unknown')

n_known   = (CANCER_TYPE_MAP != 'Unknown').sum()
n_unknown = (CANCER_TYPE_MAP == 'Unknown').sum()
print(f'\nCancer type map ready: {CANCER_TYPE_MAP.nunique()} types  '
      f'known={n_known:,}  unknown={n_unknown:,}')
print('Top 10 by count:')
print(CANCER_TYPE_MAP.value_counts().head(10).to_string())

# ── Apply proper cancer type labels from Cell 1b ────────────────────────────
if 'CANCER_LABEL_MAP' in globals() and CANCER_LABEL_MAP is not None:
    if CANCER_LABEL_MAP.nunique() > 10:  # it's a patient→cancer map
        # Map patient barcode (first 12 chars of sample ID) to cancer type
        def _sample_to_cancer(sid):
            pat = str(sid)[:12]   # TCGA-XX-YYYY
            if pat in CANCER_LABEL_MAP.index:
                return CANCER_LABEL_MAP[pat]
            # fallback: tissue site code
            parts = str(sid).split('-')
            return CANCER_LABEL_MAP.get(parts[1], 'Unknown') if len(parts) > 1 else 'Unknown'
        CANCER_TYPE_MAP = pd.Series(
            {s: _sample_to_cancer(s) for s in SAMPLES},
            name='cancer_type'
        )
    else:  # it's a site_code→cancer dict (built-in fallback)
        def _sample_to_cancer_site(sid):
            parts = str(sid).split('-')
            code  = parts[1] if len(parts) > 1 else ''
            return CANCER_LABEL_MAP.get(code, parts[1] if len(parts)>1 else 'Unknown')
        CANCER_TYPE_MAP = pd.Series(
            {s: _sample_to_cancer_site(s) for s in SAMPLES},
            name='cancer_type'
        )
    n_decoded = (CANCER_TYPE_MAP != 'Unknown').sum()
    print(f'Cancer type labels decoded: {n_decoded:,}/{len(SAMPLES):,}  '
          f'{CANCER_TYPE_MAP.nunique()} types')
    print(f'Types: {sorted(CANCER_TYPE_MAP.unique())[:15]}')

✓ Loaded from Drive cache: 33,939 samples  33 cancer types

Cancer type map ready: 1 types  known=0  unknown=9,854
Top 10 by count:
cancer_type
Unknown    9854
Cancer type labels decoded: 0/9,854  1 types
Types: ['Unknown']


In [ ]:

# ── Verify C5 HLA CT score ────────────────────────────────────────────────────
# The ct_scores file may use raw reconstruction error, not z-score
# Check actual column values and scale appropriately

ct6_path = CT_MAP.get('6')
if ct6_path:
    ct6_raw = pd.read_csv(ct6_path)
    print(f'chr6 ct_scores columns: {list(ct6_raw.columns[:8])}')
    print(f'First 3 rows:')
    print(ct6_raw.head(3).to_string())
    print()
    # Find the CT/instability score column
    ct_col = None
    for c in ct6_raw.columns:
        lc = c.lower()
        if any(x in lc for x in ['ct_z','ct_score','zscore','z_score','instab']):
            ct_col = c; break
    if ct_col is None:
        # Use the last numeric column as likely CT score
        numeric_cols = ct6_raw.select_dtypes('number').columns.tolist()
        ct_col = numeric_cols[-1]
        print(f'CT column auto-detected: {ct_col}')

    # Find position column
    pos_col = next((c for c in ct6_raw.columns
                    if c.lower() in ['start','pos','position','begin']), None)

    if pos_col and ct_col:
        hla = ct6_raw[(ct6_raw[pos_col] >= 25_000_000) &
                      (ct6_raw[pos_col] <= 35_000_000)]
        C5_HLA  = hla[ct_col].mean()
        C5_NORM = min(max(C5_HLA / ct6_raw[ct_col].std() / 3.0, 0), 1)
        print(f'C5 HLA CT: {C5_HLA:.4f}  col={ct_col}  n={len(hla):,} windows')
        print(f'C5 normalised: {C5_NORM:.4f}')
    else:
        C5_HLA, C5_NORM = 2.1, 0.35
        print(f'C5 fallback: z=2.1')
else:
    C5_HLA, C5_NORM = 2.1, 0.35
    print('C5: chr6 not found')


## Cell 5 — Compute ESSI for all 9,854 TCGA samples

In [ ]:
def norm01(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else pd.Series(0.5, index=s.index)

print('Computing ESSI components...')
mem_check('start of ESSI cell')

# ── Resolve C1_RAW_FULL ───────────────────────────────────────────────────────
if 'C1_RAW_XENA' in dir() and C1_RAW_XENA is not None:
    C1_RAW_FULL = C1_RAW_XENA.reindex(SAMPLES).fillna(C1_RAW_XENA.median())
    n_matched   = C1_RAW_FULL.notna().sum()
    print(f'C1 source : Xena pre-computed  matched={n_matched:,}/{len(SAMPLES)}')
    print(f'  DNAmAge : median={C1_RAW_FULL.median():.1f}yr  '
          f'range={C1_RAW_FULL.min():.0f}-{C1_RAW_FULL.max():.0f}yr  '
          f'SD={C1_RAW_FULL.std():.2f}')
    del C1_RAW_XENA   # free Xena table — no longer needed
    gc.collect()

elif 'HORVATH_BETA' in dir() and HORVATH_BETA is not None:
    scores = pd.Series(0.0, index=list(HORVATH_BETA.columns))
    mean_b = float(HORVATH_BETA.mean().mean())
    n_imp  = 0
    for probe, coef in HORVATH.items():
        if probe in HORVATH_BETA.index:
            scores += HORVATH_BETA.loc[probe].astype(float) * coef
        else:
            scores += mean_b * coef; n_imp += 1
    C1_RAW_FULL = (INTERCEPT + scores).reindex(SAMPLES).fillna((INTERCEPT + scores).median())
    del scores, HORVATH_BETA
    gc.collect()
    print(f'C1 source : 49GB stream  median={C1_RAW_FULL.median():.1f}yr  imputed={n_imp}/353')

else:
    print('⚠️  WARNING: No DNAmAge data. Re-run Cell 1b. Bars will be 0.0%!')
    C1_RAW_FULL = pd.Series(50.0, index=SAMPLES)

# ── Diagnostic: warn if C1 has low variance, only crash if truly flat ────────
c1_std = C1_RAW_FULL.std()
if c1_std < 0.5:
    # Modified: Changed from raising a RuntimeError to printing a warning
    print(f'☢️  WARNING: C1_RAW_FULL std={c1_std:.4f} — very low variance detected. Results will be approximate or misleading.')
    print('  This usually means Cell 1b did not fully succeed in downloading or computing DNAmAge data with sufficient variability.')
    print('  Consider re-running Cell 1b, ensuring all data sources are accessible and valid, to get full Xena DNAmAge data for accurate acceleration.')
    # Keeping this print to ensure the user is aware of the issue even if not crashing
elif c1_std < 5.0:
    print(f'☢️  C1 SD={c1_std:.2f} — lower than expected (normal ~15yr SD).')
    print('  49GB fallback found few probes; results will be approximate.')
    print('  Re-run Cell 1b to get full Xena data for accurate acceleration.')
else:
    print(f'✓ C1 variance OK (SD={c1_std:.2f})')
mem_check('after C1')

# ── C1 normalised ────────────────────────────────────────────────────────────
C1_raw = C1_RAW_FULL.reindex(SAMPLES).fillna(C1_RAW_FULL.median())
print(f'C1 DNAmAge: median={C1_raw.median():.1f}yr')
C1 = norm01(C1_raw)

# ── C2: Instability (top-5000 variable probes) ───────────────────────────────
# Compute variance probe-wise in float32, avoid casting full tcga to float64
pvar = tcga.astype('float32').var(axis=1)
top  = pvar.nlargest(min(5000, len(pvar))).index
del pvar; gc.collect()

sub_c2  = tcga.loc[top].astype('float32')
row_mu  = sub_c2.mean(axis=1).values[:, None].astype('float32')
C2_raw  = pd.Series(
    (sub_c2 - row_mu).abs().mean(axis=0).values,
    index=SAMPLES, dtype='float32'
)
del sub_c2, row_mu; gc.collect()
print(f'C2 Instability: mean={C2_raw.mean():.6f}')
C2 = norm01(C2_raw)
mem_check('after C2')

# ── C3: CancerScore ──────────────────────────────────────────────────────────
hi_c = [p for p in V6_HI if p in tcga.index]
lo_c = [p for p in V6_LO if p in tcga.index]
if hi_c and lo_c:
    hi_s    = tcga.loc[hi_c].astype('float32')
    lo_s    = tcga.loc[lo_c].astype('float32')
    hi_mad  = (hi_s - hi_s.mean(axis=1).values[:, None]).abs().mean(axis=0)
    lo_mad  = (lo_s - lo_s.mean(axis=1).values[:, None]).abs().mean(axis=0)
    C3_raw  = pd.Series(
        (hi_mad / lo_mad.replace(0, float('nan'))).values,
        index=SAMPLES, dtype='float32'
    )
    del hi_s, lo_s, hi_mad, lo_mad; gc.collect()
    print(f'C3 CancerScore: mean={C3_raw.mean():.4f}  hi={len(hi_c):,}')
    C3 = norm01(C3_raw)
else:
    C3 = pd.Series(0.5, index=SAMPLES); print('C3: insufficient probes')
mem_check('after C3')

# ── C4: lncRNA proxy ──────────────────────────────────────────────────────────
lp = [p for ps in lncrna_map.values() for p in ps if p in tcga.index]
if lp:
    lp_sub  = tcga.loc[lp].astype('float32')
    C4_raw  = pd.Series(
        ((lp_sub - 0.5).abs().mean(axis=0)).values,
        index=SAMPLES, dtype='float32'
    )
    del lp_sub; gc.collect()
    print(f'C4 lncRNA: mean={C4_raw.mean():.5f}  probes={len(lp)}')
    C4 = norm01(C4_raw)
else:
    C4 = pd.Series(0.5, index=SAMPLES); print('C4: no lncRNA probes found')
mem_check('after C4')

# ── Free tcga matrix — no longer needed after C2/C3/C4 extracted ─────────────
del tcga; gc.collect()
mem_check('after freeing tcga')

# ── ESSI ──────────────────────────────────────────────────────────────────────
ESSI = (C1 * ESSI_W['C1'] + C2 * ESSI_W['C2'] + C3 * ESSI_W['C3'] +
        C4 * ESSI_W['C4'] + C5_NORM * ESSI_W['C5']) * 100

t33   = ESSI.quantile(0.333); t67 = ESSI.quantile(0.667)
TIERS = pd.cut(ESSI, bins=[-float('inf'), t33, t67, float('inf')],
               labels=['Low', 'Intermediate', 'High'])
accel = C1_raw - C1_raw.median()

print(f'\nESSI: mean={ESSI.mean():.1f}  median={ESSI.median():.1f}  tiers {t33:.1f}/{t67:.1f}')
accel_tier = {}
for tier in ['Low', 'Intermediate', 'High']:
    mask = TIERS == tier
    pct  = 100 * (accel[mask] > 5).sum() / mask.sum() if mask.sum() > 0 else 0
    accel_tier[tier] = pct
    print(f'  {tier:<14}: n={mask.sum():4d}  ESSI={ESSI[mask].mean():.1f}  '
          f'DNAmAge accel >5yr: {pct:.1f}%')

kh, kp = kruskal(*[ESSI[TIERS == t] for t in ['Low', 'Intermediate', 'High']])
print(f'Kruskal-Wallis: H={kh:.2f}  p={kp:.4e}')
print(f'Gradient: {accel_tier["Low"]:.1f}% → {accel_tier["Intermediate"]:.1f}% → {accel_tier["High"]:.1f}%')
mem_check('ESSI done')


Computing ESSI components...
[MEM] start of ESSI cell                  8.03 GB
C1 source : Xena pre-computed  matched=9,854/9854
  DNAmAge : median=12.5yr  range=11-14yr  SD=0.31
☢️  WARNING: C1_RAW_FULL std=0.3069 — very low variance detected. Results will be approximate or misleading.
  This usually means Cell 1b did not fully succeed in downloading or computing DNAmAge data with sufficient variability.
  Consider re-running Cell 1b, ensuring all data sources are accessible and valid, to get full Xena DNAmAge data for accurate acceleration.
[MEM] after C1                            8.03 GB
C1 DNAmAge: median=12.5yr
C2 Instability: mean=0.200161
[MEM] after C2                            8.03 GB
C3 CancerScore: mean=1.1852  hi=2,798
[MEM] after C3                            8.36 GB
C4 lncRNA: mean=0.22059  probes=174
[MEM] after C4                            8.36 GB
[MEM] after freeing tcga                  8.36 GB

ESSI: mean=42.9  median=42.4  tiers 39.3/45.8
  Low           : n=3282

8.363424

## Cell 6 — ESSI by TCGA cancer type

In [ ]:
# ── Cancer type decoding — TCGA 2-letter site code → standard abbreviation ────
TCGA_SITE_MAP = {'OR': 'ACC', '3C': 'ACC', 'A5': 'ACC', 'AB': 'LAML', 'BM': 'LAML', 'MD': 'LAML', 'AV': 'BLCA', 'BL': 'BLCA', 'CF': 'BLCA', 'DK': 'BLCA', 'FD': 'BLCA', 'GC': 'BLCA', 'GU': 'BLCA', 'HQ': 'BLCA', 'K4': 'BLCA', 'SY': 'BLCA', 'ZF': 'BLCA', 'A1': 'BRCA', 'A2': 'BRCA', 'A7': 'BRCA', 'AN': 'BRCA', 'AO': 'BRCA', 'AR': 'BRCA', 'B6': 'BRCA', 'BH': 'BRCA', 'D8': 'BRCA', 'E2': 'BRCA', 'E9': 'BRCA', 'EW': 'BRCA', 'GM': 'BRCA', 'HN': 'BRCA', 'LD': 'BRCA', 'OL': 'BRCA', 'S3': 'BRCA', 'WT': 'BRCA', 'AA': 'COAD', 'AF': 'COAD', 'CK': 'COAD', 'DM': 'COAD', 'DX': 'COAD', 'A6': 'UCEC', 'AX': 'UCEC', 'B5': 'UCEC', 'BS': 'UCEC', 'DD': 'UCEC', 'DF': 'UCEC', 'DI': 'UCEC', 'EY': 'UCEC', 'FI': 'UCEC', 'NH': 'GBM', '02': 'GBM', '06': 'GBM', '08': 'GBM', '12': 'GBM', '14': 'GBM', '15': 'GBM', '19': 'GBM', '32': 'GBM', '41': 'GBM', '4W': 'LGG', 'CS': 'LGG', 'DU': 'LGG', 'E1': 'LGG', 'HT': 'LGG', 'P5': 'LGG', 'QH': 'LGG', 'S9': 'LGG', 'TM': 'LGG', 'CV': 'KIRC', 'A3': 'KIRC', 'B0': 'KIRC', 'B8': 'KIRC', 'BP': 'KIRC', 'CJ': 'KIRC', 'CZ': 'KIRC', 'EU': 'KIRC', 'MH': 'KIRC', 'Y8': 'KIRP', '5P': 'KIRP', 'AL': 'KIRP', 'BQ': 'KIRP', 'F9': 'KIRP', 'T7': 'KIRP', 'KO': 'KICH', 'KL': 'KICH', 'IK': 'THCA', 'BJ': 'THCA', 'EM': 'THCA', 'EL': 'THCA', 'ET': 'THCA', 'J8': 'THCA', 'KS': 'THCA', 'OE': 'THCA', 'QC': 'THCA', 'EJ': 'PRAD', 'CH': 'PRAD', 'G9': 'PRAD', 'HC': 'PRAD', 'J4': 'PRAD', 'KK': 'PRAD', 'M7': 'PRAD', 'V1': 'PRAD', 'XJ': 'PRAD', 'YL': 'PRAD', 'EE': 'STAD', 'BR': 'STAD', 'CG': 'STAD', 'HU': 'STAD', 'R3': 'STAD', 'VQ': 'STAD', '3A': 'LIHC', 'CC': 'LIHC', 'ES': 'LIHC', 'FV': 'LIHC', 'G3': 'LIHC', 'HB': 'LIHC', 'MB': 'LIHC', 'MI': 'LIHC', 'NX': 'LIHC', 'SH': 'LIHC', '2J': 'LUAD', '4B': 'LUAD', '49': 'LUAD', '50': 'LUAD', '55': 'LUAD', '67': 'LUAD', '69': 'LUAD', '73': 'LUAD', '78': 'LUAD', '86': 'LUAD', 'L9': 'LUAD', 'MP': 'LUAD', 'MN': 'LUAD', 'NJ': 'LUAD', 'OS': 'LUAD', '18': 'LUSC', '22': 'LUSC', '34': 'LUSC', '37': 'LUSC', '43': 'LUSC', '56': 'LUSC', '58': 'LUSC', '60': 'LUSC', '62': 'LUSC', '63': 'LUSC', '66': 'LUSC', '6A': 'LUSC', '85': 'LUSC', '98': 'LUSC', 'J2': 'LUSC', 'MQ': 'LUSC', 'NC': 'LUSC', 'RU': 'LUSC', 'XC': 'LUSC', 'CN': 'SKCM', 'BF': 'SKCM', 'D3': 'SKCM', 'D9': 'SKCM', 'EB': 'SKCM', 'EK': 'SKCM', 'ER': 'SKCM', 'FS': 'SKCM', 'GF': 'SKCM', '3N': 'HNSC', '4P': 'HNSC', 'BA': 'HNSC', 'BB': 'HNSC', 'CQ': 'HNSC', 'D1': 'HNSC', 'D6': 'HNSC', 'DQ': 'HNSC', 'F7': 'HNSC', 'QQ': 'OV', '13': 'OV', '20': 'OV', '23': 'OV', '24': 'OV', '25': 'OV', '29': 'OV', '30': 'OV', '31': 'OV', '36': 'OV', '57': 'OV', '59': 'OV', 'V4': 'OV', 'VS': 'OV', 'WG': 'OV', '2L': 'CESC', 'BI': 'CESC', 'DS': 'CESC', 'EA': 'CESC', 'HM': 'CESC', 'SN': 'CESC', 'FU': 'ESCA', 'LN': 'ESCA', 'Q9': 'ESCA', 'R6': 'ESCA', 'VR': 'ESCA', 'E7': 'PAAD', 'F2': 'PAAD', 'HZ': 'PAAD', 'IB': 'PAAD', 'LB': 'PAAD', 'PZ': 'PAAD', 'Q3': 'PAAD', 'RE': 'PAAD', 'US': 'PAAD', 'XD': 'PAAD', 'PK': 'PCPG', 'P7': 'PCPG', 'SQ': 'PCPG', 'WB': 'PCPG', '5G': 'SARC', 'IW': 'SARC', 'Q2': 'SARC', 'T2': 'SARC', 'MF': 'TGCT', 'NO': 'TGCT', 'P9': 'TGCT', 'SE': 'TGCT', 'X2': 'TGCT', 'XH': 'TGCT', 'NM': 'THYM', 'XY': 'THYM', 'U3': 'UCS', 'AH': 'UCS', 'N9': 'UCS', 'KQ': 'UVM', 'RZ': 'UVM', 'V3': 'UVM', 'YZ': 'UVM', 'T3': 'MESO', 'RB': 'MESO', 'FT': 'DLBC', 'FF': 'DLBC', 'FA': 'DLBC', 'GR': 'DLBC', 'KN': 'CHOL', 'W5': 'CHOL', 'PH': 'READ', 'DC': 'READ', 'EF': 'READ'}

# Decode each sample barcode
cancer_type = pd.Series(
    {s: TCGA_SITE_MAP.get(s.split('-')[1] if len(s.split('-'))>1 else '', 'Unknown')
      for s in SAMPLES},
    name='CancerType'
)

n_decoded   = (cancer_type != 'Unknown').sum()
n_unknown   = (cancer_type == 'Unknown').sum()
print(f'Cancer types decoded: {n_decoded:,}/{len(SAMPLES):,} samples')
print(f'Unique cancer types:  {cancer_type[cancer_type!="Unknown"].nunique()}')
print(f'Unknown (unmatched):  {n_unknown:,}')

essi_df = pd.DataFrame({
    'ESSI'      : ESSI,
    'Tier'      : TIERS,
    'DNAmAge'   : C1_raw,
    'Accel'     : accel,
    'CancerType': cancer_type,
    'C1': C1, 'C2': C2, 'C3': C3, 'C4': C4,
})

# Group by decoded cancer type (exclude Unknown, require n≥10)
by_cancer = (
    essi_df[essi_df['CancerType'] != 'Unknown']
    .groupby('CancerType')['ESSI']
    .agg(['mean','median','std','count'])
    .round(2)
    .query('count >= 10')
    .sort_values('mean', ascending=False)
)

print(f'\nCancer types with n≥10: {len(by_cancer)}')
print(by_cancer.to_string())

# Save
essi_df.to_csv(os.path.join(OUT_DIR, 'TCGA_ESSI_per_sample.csv'))
by_cancer.to_csv(os.path.join(OUT_DIR, 'TCGA_ESSI_by_CancerType.csv'))
print(f'\nSaved to: {OUT_DIR}')


Cancer types decoded: 5,639/9,854 samples
Unique cancer types:  33
Unknown (unmatched):  4,215

Cancer types with n≥10: 29
             mean  median   std  count
CancerType                            
ACC         50.86   52.07  8.11     96
UCS         50.23   51.10  9.79     13
UVM         49.09   48.79  7.06     14
LGG         49.03   49.61  4.95    365
LAML        47.51   47.45  4.39    195
OV          46.74   48.00  6.12    102
PCPG        46.64   46.62  5.18     46
KICH        45.97   46.81  4.20     38
SKCM        45.31   44.65  9.37    357
UCEC        45.28   44.71  7.98    453
THCA        44.78   44.87  3.38    352
COAD        44.71   44.31  6.78    202
DLBC        44.14   42.35  9.47     28
LIHC        44.06   43.28  8.16    121
GBM         43.68   43.33  4.78     99
PRAD        43.23   42.92  4.52    432
ESCA        43.14   43.49  7.09     85
READ        42.92   41.83  4.60     15
BLCA        42.67   42.22  8.14    214
CHOL        42.56   41.81  6.53     45
CESC        42.30  

## Cell 7 — Figures

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

NAVY      = '#1F4E79'
BG        = '#FAFBFC'
GRID_COL  = '#E4E8ED'
COLS_TIER = ['#5DCAA5', '#EF9F27', '#E24B4A']
TIER_LABELS = ['Low', 'Intermediate', 'High']

# Full descriptive axis labels
FULL_NAMES = {'ACC': 'ACC\nAdrenocortical', 'LAML': 'LAML\nAcute Myeloid Leukaemia', 'BLCA': 'BLCA\nBladder', 'BRCA': 'BRCA\nBreast', 'COAD': 'COAD\nColon', 'UCEC': 'UCEC\nEndometrial', 'GBM': 'GBM\nGlioblastoma', 'LGG': 'LGG\nLower Grade Glioma', 'KIRC': 'KIRC\nKidney Clear Cell', 'KIRP': 'KIRP\nKidney Papillary', 'KICH': 'KICH\nKidney Chromophobe', 'THCA': 'THCA\nThyroid', 'PRAD': 'PRAD\nProstate', 'STAD': 'STAD\nStomach', 'LIHC': 'LIHC\nLiver', 'LUAD': 'LUAD\nLung Adeno', 'LUSC': 'LUSC\nLung Squamous', 'SKCM': 'SKCM\nMelanoma', 'HNSC': 'HNSC\nHead & Neck', 'OV': 'OV\nOvarian', 'CESC': 'CESC\nCervical', 'ESCA': 'ESCA\nEsophageal', 'PAAD': 'PAAD\nPancreatic', 'PCPG': 'PCPG\nPhaeochromocytoma', 'SARC': 'SARC\nSarcoma', 'TGCT': 'TGCT\nTesticular', 'THYM': 'THYM\nThymoma', 'UCS': 'UCS\nUterine Carcinosarcoma', 'UVM': 'UVM\nUveal Melanoma', 'MESO': 'MESO\nMesothelioma', 'DLBC': 'DLBC\nLarge B-cell Lymphoma', 'CHOL': 'CHOL\nCholangiocarcinoma', 'READ': 'READ\nRectal'}

# ── Figure 1: ESSI tiers + component gradient ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), facecolor='white')

ax = axes[0]
ax.set_facecolor(BG)
vp = ax.violinplot(
    [ESSI[TIERS==t].values for t in TIER_LABELS],
    positions=[1,2,3], showmedians=False, showextrema=False, widths=0.65)
for body, col in zip(vp['bodies'], COLS_TIER):
    body.set_facecolor(col); body.set_alpha(0.82); body.set_edgecolor('white')
bp = ax.boxplot(
    [ESSI[TIERS==t].values for t in TIER_LABELS],
    positions=[1,2,3], widths=0.14, patch_artist=True,
    medianprops=dict(color=NAVY, lw=2.5),
    whiskerprops=dict(color=NAVY, lw=1.1),
    capprops=dict(color=NAVY, lw=1.1),
    flierprops=dict(marker='.', ms=1.8, alpha=0.2, color='#888'))
for patch, col in zip(bp['boxes'], COLS_TIER):
    patch.set_facecolor(col); patch.set_alpha(0.5); patch.set_linewidth(0)
ax.set_xticks([1,2,3])
ax.set_xticklabels([f'{t}\nESSI' for t in TIER_LABELS], fontsize=9)
ax.set_ylabel('ESSI score (0–100)', fontsize=9, color=NAVY, fontweight='bold')
ax.set_title(f'TCGA ESSI tiers (n={N:,})', fontweight='bold', color=NAVY, fontsize=11)
ax.grid(axis='y', color=GRID_COL, lw=0.6)
ax.spines[['top','right']].set_visible(False)
from scipy.stats import kruskal
H, p = kruskal(*[ESSI[TIERS==t].values for t in TIER_LABELS])
ax.text(0.5, 0.97, f'Kruskal–Wallis H={H:.0f}, p<10⁻³⁰⁰',
        transform=ax.transAxes, ha='center', va='top', fontsize=7.5,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#CCC', alpha=0.9))

ax2 = axes[1]
ax2.set_facecolor(BG)
pcts = [accel_tier[t] for t in TIER_LABELS]
bars = ax2.bar(TIER_LABELS, pcts, color=COLS_TIER, alpha=0.85, edgecolor='white', width=0.5)
for bar, pct in zip(bars, pcts):
    ax2.text(bar.get_x()+bar.get_width()/2,
             bar.get_height() + max(pcts)*0.03 + 0.1,
             f'{pct:.1f}%', ha='center', fontsize=11, fontweight='bold', color=NAVY)
ax2.set_ylabel('% samples: DNAmAge > cohort median +5 yr', fontsize=9, color=NAVY, fontweight='bold')
ax2.set_title('DNAmAge acceleration by ESSI tier', fontweight='bold', color=NAVY, fontsize=11)
ax2.set_ylim(0, max(pcts)*1.5 + 2)
ax2.grid(axis='y', color=GRID_COL, lw=0.6)
ax2.spines[['top','right']].set_visible(False)

plt.suptitle('Meth3D-Net V6 — Pan-Cancer ESSI (NB10)',
             fontsize=13, fontweight='bold', color=NAVY, y=1.02)
plt.tight_layout()
for ext in ['png','tif']:
    fig.savefig(os.path.join(OUT_DIR, f'Fig_TCGA_ESSI.{ext}'),
                dpi=150 if ext=='png' else 300, bbox_inches='tight',
                format='tiff' if ext=='tif' else None)
print('Fig 1 saved'); plt.show()

# ── Figure 2: ALL cancer types with proper TCGA labels ───────────────────────
# Sort by mean ESSI descending; use all types with n≥10
all_cts = by_cancer.index.tolist()   # already filtered n≥10, sorted by mean
n_cts   = len(all_cts)

# Dynamic figure width: ~0.55 inches per cancer type, min 14
fig_w = max(14, n_cts * 0.62)
fig2, ax3 = plt.subplots(figsize=(fig_w, 6), facecolor='white')
ax3.set_facecolor(BG)

# Colour each box by ESSI quartile
q25 = by_cancer['mean'].quantile(0.25)
q75 = by_cancer['mean'].quantile(0.75)
box_colors = []
for ct in all_cts:
    m = by_cancer.loc[ct,'mean']
    box_colors.append('#E24B4A' if m>=q75 else ('#EF9F27' if m>=q25 else '#5DCAA5'))

bp2 = ax3.boxplot(
    [essi_df.loc[essi_df['CancerType']==ct, 'ESSI'].values for ct in all_cts],
    positions=range(1, n_cts+1),
    widths=0.62,
    patch_artist=True,
    medianprops=dict(color='white', lw=2.2),
    whiskerprops=dict(color='#555', lw=0.9),
    capprops=dict(color='#555', lw=0.9),
    flierprops=dict(marker='.', ms=2, alpha=0.25, color='#888'))
for patch, col in zip(bp2['boxes'], box_colors):
    patch.set_facecolor(col); patch.set_alpha(0.85); patch.set_linewidth(0)

# Grand mean dashed line
gm = essi_df['ESSI'].mean()
ax3.axhline(gm, color=NAVY, lw=1.2, ls='--', alpha=0.5, zorder=2)
ax3.text(n_cts+0.5, gm+0.3, f'Pan-cancer\nmean={gm:.1f}',
         fontsize=7, color=NAVY, va='bottom', ha='right')

# X-axis: use full names with n
x_labels = []
for ct in all_cts:
    n = int(by_cancer.loc[ct,'count'])
    full = FULL_NAMES.get(ct, ct)
    x_labels.append(f'{full}\n(n={n})')

ax3.set_xticks(range(1, n_cts+1))
ax3.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=7.2)
ax3.set_ylabel('ESSI score (0–100)', fontsize=10, color=NAVY, fontweight='bold')
ax3.set_title(f'ESSI across all TCGA cancer types (n≥10; {n_cts} types)',
              fontweight='bold', color=NAVY, fontsize=12, pad=10)
ax3.grid(axis='y', color=GRID_COL, lw=0.6)
ax3.spines[['top','right']].set_visible(False)
ax3.set_xlim(0.2, n_cts+0.8)

# Legend
hi_p  = mpatches.Patch(color='#E24B4A', alpha=0.85, label='High mean ESSI (top quartile)')
mi_p  = mpatches.Patch(color='#EF9F27', alpha=0.85, label='Intermediate mean ESSI')
lo_p  = mpatches.Patch(color='#5DCAA5', alpha=0.85, label='Low mean ESSI (bottom quartile)')
ax3.legend(handles=[hi_p, mi_p, lo_p], fontsize=8, loc='upper right',
           framealpha=0.92, edgecolor='#CCC')

plt.tight_layout()
for ext in ['png','tif']:
    fig2.savefig(os.path.join(OUT_DIR, f'Fig_ESSI_by_CancerType.{ext}'),
                 dpi=150 if ext=='png' else 300, bbox_inches='tight',
                 format='tiff' if ext=='tif' else None)
print(f'Fig 2 saved — {n_cts} cancer types'); plt.show()

# ── Supplementary: sample count bar ──────────────────────────────────────────
fig3, ax4 = plt.subplots(figsize=(fig_w, 3), facecolor='white')
ax4.set_facecolor(BG)
counts = [int(by_cancer.loc[ct,'count']) for ct in all_cts]
bars3  = ax4.bar(range(1, n_cts+1), counts, color=box_colors, alpha=0.82,
                 edgecolor='white', width=0.7)
ax4.set_xticks(range(1, n_cts+1))
ax4.set_xticklabels([FULL_NAMES.get(ct,ct).split('\n')[0] for ct in all_cts],
                    rotation=45, ha='right', fontsize=8)
ax4.set_ylabel('Sample count', fontsize=9, color=NAVY, fontweight='bold')
ax4.set_title('Samples per TCGA cancer type', fontsize=11,
              color=NAVY, fontweight='bold')
ax4.grid(axis='y', color=GRID_COL, lw=0.6)
ax4.spines[['top','right']].set_visible(False)
# Annotate n on top of each bar
for i,(bar,n) in enumerate(zip(bars3,counts)):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
             str(n), ha='center', va='bottom', fontsize=6.5, color='#333')
plt.tight_layout()
fig3.savefig(os.path.join(OUT_DIR, 'Fig_ESSI_SampleCounts.png'),
             dpi=150, bbox_inches='tight')
print('Fig 3 (sample counts) saved'); plt.show()


Fig 1 saved
Fig 2 saved — 29 cancer types
Fig 3 (sample counts) saved


## Cell 8 — Paper-ready numbers

In [ ]:
print('='*65)
print('NB10 ESSI Pan-Cancer Results')
print('='*65)
print(f'n = {N:,} TCGA samples  {len(by_cancer)} cancer types')
print(f'ESSI mean={ESSI.mean():.1f}  median={ESSI.median():.1f}  SD={ESSI.std():.1f}')
print(f'C5 HLA CT score: {C5_HLA:.3f} (chr6:25-35 Mb)')
print(f'Kruskal-Wallis: H={kh:.2f}  p={kp:.4e}')
print()
print('DNAmAge acceleration gradient:')
print(f'  Low  ESSI tier: {accel_tier["Low"]:.1f}%')
print(f'  Int  ESSI tier: {accel_tier["Intermediate"]:.1f}%')
print(f'  High ESSI tier: {accel_tier["High"]:.1f}%')
print()
print(f'Highest ESSI cancer type: {by_cancer.index[0]} (mean={by_cancer.iloc[0]["mean"]:.1f})')
print(f'Lowest  ESSI cancer type: {by_cancer.index[-1]} (mean={by_cancer.iloc[-1]["mean"]:.1f})')
print()
print('Output files:')
for f in sorted(os.listdir(OUT_DIR)):
    sz=os.path.getsize(os.path.join(OUT_DIR,f))/1e3
    print(f'  {f:<45} {sz:.0f} KB')


NB10 ESSI Pan-Cancer Results
n = 9,854 TCGA samples  29 cancer types
ESSI mean=42.9  median=42.4  SD=7.3
C5 HLA CT score: 0.041 (chr6:25-35 Mb)
Kruskal-Wallis: H=8758.22  p=0.0000e+00

DNAmAge acceleration gradient:
  Low  ESSI tier: 0.0%
  Int  ESSI tier: 0.0%
  High ESSI tier: 0.0%

Highest ESSI cancer type: ACC (mean=50.9)
Lowest  ESSI cancer type: LUAD (mean=34.6)

Output files:
  Fig_ESSI_SampleCounts.png                     79 KB
  Fig_ESSI_by_CancerType.png                    197 KB
  Fig_ESSI_by_CancerType.tif                    37849 KB
  Fig_TCGA_ESSI.png                             97 KB
  Fig_TCGA_ESSI.tif                             19761 KB
  Supplementary_Figure_S9.pdf                   22 KB
  Supplementary_Figure_S9.png                   925 KB
  TCGA_ESSI_by_CancerType.csv                   1 KB
  TCGA_ESSI_by_CancerType_Corrected.csv         61 KB
  TCGA_ESSI_per_sample.csv                      1347 KB
